# VoxCPM2 Dual‑T4 Voice Studio — Final UI

Kaggle-focused Burmese voice studio with two T4 workers, long-form chunking, 12 voice-design profiles, persistent saved voices, remote/direct voice cloning, optional cleanup, Whisper transcription, deterministic speed control, and exact chunk-join pauses.

**Important:** restart the Kaggle session once before switching from the older notebook version so old GPU worker processes do not remain in VRAM.


In [1]:
# Cell 1 — install Voice Studio dependencies
!pip -q install -U \
    voxcpm soundfile kagglehub huggingface_hub \
    yt-dlp gdown faster-whisper noisereduce \
    mega.py-v2 demucs

# Kaggle normally has ffmpeg; install it only when missing.
!which ffmpeg >/dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq ffmpeg)

print("✓ Voice Studio dependencies installed")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 11.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 2.7 MB/s eta 0:00:00
   ━━━━

In [2]:
# Cell 2 — download model once to a shared local directory
from huggingface_hub import snapshot_download
import os

MODEL_DIR = "/kaggle/working/VoxCPM2"

if not os.path.exists(os.path.join(MODEL_DIR, "model.safetensors")):
    print("Downloading VoxCPM2...")
    snapshot_download(
        repo_id="openbmb/VoxCPM2",
        local_dir=MODEL_DIR,
    )
else:
    print("✓ VoxCPM2 already exists in", MODEL_DIR)

print("✓ Model directory:", MODEL_DIR)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

✓ Model directory: /kaggle/working/VoxCPM2


In [3]:
%%writefile /kaggle/working/voxcpm_dual_worker.py
import os
import random
import inspect
import traceback

def worker_main(physical_gpu_id, model_dir, task_q, result_q, optimize=False):
    # One physical T4 is exposed to each child. Inside the child, "cuda" is correct.
    os.environ["CUDA_VISIBLE_DEVICES"] = str(physical_gpu_id)

    try:
        import numpy as np
        import torch
        import soundfile as sf
        from voxcpm import VoxCPM

        result_q.put({
            "status": "loading",
            "gpu": physical_gpu_id,
        })

        model = VoxCPM.from_pretrained(
            model_dir,
            load_denoiser=False,
            optimize=bool(optimize),
            device="cuda",
        )

        try:
            sample_rate = int(model.tts_model.sample_rate)
        except Exception:
            sample_rate = 48000

        # Detect the parameters accepted by the *installed* VoxCPM build.
        try:
            sig = inspect.signature(model._generate)
            params = sig.parameters
            accepts_any_kwarg = any(
                p.kind == inspect.Parameter.VAR_KEYWORD
                for p in params.values()
            )
            supported = set(params.keys())
        except Exception:
            accepts_any_kwarg = True
            supported = set()

        result_q.put({
            "status": "ready",
            "gpu": physical_gpu_id,
            "sample_rate": sample_rate,
            "supported_params": sorted(supported),
        })

    except Exception as e:
        result_q.put({
            "status": "startup_error",
            "gpu": physical_gpu_id,
            "error": repr(e),
            "traceback": traceback.format_exc(),
        })
        return

    while True:
        task = task_q.get()

        if task is None:
            break

        batch_id = task["batch_id"]
        task_id = task["task_id"]
        chunk_index = task.get("chunk_index", 0)
        output_path = task["output_path"]
        kwargs = dict(task["kwargs"])

        try:
            os.makedirs(os.path.dirname(output_path), exist_ok=True)

            # ---------------------------------------------------------
            # VERSION-COMPATIBILITY LAYER
            # ---------------------------------------------------------
            # Recent VoxCPM builds accept seed= in _generate().
            # The Kaggle build in this user's runtime does not.
            #
            # If seed isn't supported, set Python/NumPy/PyTorch RNGs
            # manually, then REMOVE seed before model.generate().
            # ---------------------------------------------------------
            requested_seed = kwargs.get("seed", None)

            if (
                requested_seed is not None
                and not accepts_any_kwarg
                and "seed" not in supported
            ):
                requested_seed = int(requested_seed)

                random.seed(requested_seed)
                np.random.seed(requested_seed % (2**32 - 1))
                torch.manual_seed(requested_seed)

                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(requested_seed)

                kwargs.pop("seed", None)

            # Filter every unsupported UI option too. This makes the notebook
            # resilient if Kaggle has an older VoxCPM than the current source.
            dropped_kwargs = []

            if not accepts_any_kwarg:
                for key in list(kwargs.keys()):
                    if key not in supported:
                        dropped_kwargs.append(key)
                        kwargs.pop(key, None)

            wav = model.generate(**kwargs)

            sf.write(
                output_path,
                wav,
                sample_rate,
                subtype="PCM_16",
            )

            result_q.put({
                "status": "ok",
                "batch_id": batch_id,
                "task_id": task_id,
                "chunk_index": chunk_index,
                "gpu": physical_gpu_id,
                "output_path": output_path,
                "sample_rate": sample_rate,
                "samples": int(len(wav)),
                "duration": float(len(wav) / sample_rate),
                "dropped_kwargs": dropped_kwargs,
            })

        except Exception as e:
            result_q.put({
                "status": "error",
                "batch_id": batch_id,
                "task_id": task_id,
                "chunk_index": chunk_index,
                "gpu": physical_gpu_id,
                "error": repr(e),
                "traceback": traceback.format_exc(),
            })


Writing /kaggle/working/voxcpm_dual_worker.py


In [4]:
# ============================================================
# CELL 4 — VoxCPM2 Dual-T4 Voice Studio v6
# Burmese-safe long-form generation + continuous chunk streaming
#
# Assumes:
#   • MODEL_DIR exists (normally /kaggle/working/VoxCPM2)
#   • Cell 3 created /kaggle/working/voxcpm_dual_worker.py
#     containing worker_main(...)
# ============================================================

import os
import re
import sys
import gc
import json
import glob
import time
import uuid
import base64
import shutil
import queue as pyqueue
import threading
import subprocess
import unicodedata
from pathlib import Path
from multiprocessing import get_context
from urllib.parse import urlparse

import numpy as np
import soundfile as sf
import ipywidgets as widgets

from IPython.display import (
    display,
    Audio,
    FileLink,
    HTML,
    Javascript,
    clear_output,
)

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

MODEL_DIR = globals().get(
    "MODEL_DIR",
    "/kaggle/working/VoxCPM2",
)

VOICE_DATASET_HANDLE_DEFAULT = "cyawzin/voxcpm-voice-library"

# ------------------------------------------------------------
# HUGGING FACE / ASR DISK SETTINGS
# ------------------------------------------------------------
# Keep all large ASR downloads on Kaggle's working disk.
# Disable hf_xet because its reconstruction/cache path can temporarily
# consume another full copy of multi-GB model shards and trigger:
#   "File reconstruction error: IO Error: No space left on device"
HF_CACHE_ROOT = Path(
    "/kaggle/working/hf_cache"
)
HF_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

os.environ["HF_HOME"] = str(
    HF_CACHE_ROOT
)
os.environ["HF_HUB_CACHE"] = str(
    HF_CACHE_ROOT / "hub"
)
os.environ["TRANSFORMERS_CACHE"] = str(
    HF_CACHE_ROOT / "transformers"
)
os.environ["HF_HUB_DISABLE_XET"] = "1"


# Keep torch.compile off for T4 stability.
WORKER_OPTIMIZE = False
WORKER_STARTUP_TIMEOUT = 900

LIBRARY_ROOT = Path("/kaggle/working/voxcpm_voice_library")
VOICE_DIR = LIBRARY_ROOT / "voices"
GENERATED_DIR = LIBRARY_ROOT / "generated"
IMPORT_DIR = LIBRARY_ROOT / "imports"
COOKIE_DIR = LIBRARY_ROOT / "cookies"
META_FILE = LIBRARY_ROOT / "library.json"

for _p in (
    LIBRARY_ROOT,
    VOICE_DIR,
    GENERATED_DIR,
    IMPORT_DIR,
    COOKIE_DIR,
):
    _p.mkdir(parents=True, exist_ok=True)

LIBRARY_LOCK = threading.Lock()

# ------------------------------------------------------------
# VOICE DESIGN PROFILES
# ------------------------------------------------------------

VOICE_PROFILES = {
    "u_aung": {
        "name": "👨 U Aung — Deep Calm Male",
        "design": (
            "An adult Burmese man in his late thirties to mid forties. "
            "A naturally low baritone voice with warm chest resonance, "
            "calm confidence, mature presence, natural Burmese rhythm, "
            "clean articulation and relaxed medium-slow pacing."
        ),
    },
    "ko_min": {
        "name": "👨 Ko Min — Friendly Young Male",
        "design": (
            "A Burmese man in his mid to late twenties with a friendly, "
            "approachable medium-pitched voice. Warm, relaxed, youthful, "
            "conversational and natural with smooth phrasing."
        ),
    },
    "u_htet": {
        "name": "👴 U Htet — Wise Older Male",
        "design": (
            "A Burmese man around sixty with a mature deep voice, gentle age "
            "texture and warmth. Patient, reassuring storyteller delivery "
            "with measured pacing and thoughtful pauses."
        ),
    },
    "ko_zaw": {
        "name": "👨 Ko Zaw — Energetic Presenter",
        "design": (
            "A Burmese man around thirty with a clear, energetic and confident "
            "speaking voice. Lively but controlled, crisp articulation and "
            "slightly quick pacing without shouting."
        ),
    },
    "u_kyaw": {
        "name": "👔 U Kyaw — Formal Professional Male",
        "design": (
            "A professional Burmese man in his forties with a composed, polished "
            "and trustworthy voice. Precise articulation, stable volume, "
            "confident phrasing and a measured medium pace."
        ),
    },
    "daw_may": {
        "name": "👩 Daw May — Warm Adult Female",
        "design": (
            "A Burmese woman in her late thirties to mid forties with a warm, "
            "gentle, naturally feminine voice. Caring, grounded and trustworthy "
            "with soft but clear articulation."
        ),
    },
    "ma_su": {
        "name": "👩 Ma Su — Bright Young Female",
        "design": (
            "A Burmese woman in her early to mid twenties with a youthful, "
            "bright and naturally pleasant voice. Cheerful, friendly and "
            "expressive without sounding childish."
        ),
    },
    "daw_thida": {
        "name": "👩 Daw Thida — Elegant Mature Female",
        "design": (
            "A mature Burmese woman with a smooth, composed and elegant voice. "
            "Refined, confident, warm and professionally controlled."
        ),
    },
    "ma_nandar": {
        "name": "👩 Ma Nandar — Soft Emotional Female",
        "design": (
            "A Burmese woman in her late twenties to early thirties with a "
            "soft, intimate and emotionally expressive voice. Personal, sincere "
            "and clearly audible with gentle emotional variation."
        ),
    },
    "news_presenter": {
        "name": "📰 Burmese News Presenter",
        "design": (
            "A professional adult Burmese news presenter with a neutral, "
            "trustworthy and highly intelligible broadcast voice. Precise "
            "pronunciation, stable volume and controlled sentence emphasis."
        ),
    },
    "documentary": {
        "name": "🎥 Deep Documentary Narrator",
        "design": (
            "A mature Burmese documentary narrator with a naturally deep, "
            "resonant voice. Intelligent, observant and serious with controlled "
            "low-register resonance and subtle suspense."
        ),
    },
    "movie_recap": {
        "name": "🎬 Fast Movie Recap Narrator",
        "design": (
            "A young adult Burmese male movie-recap narrator with a natural, "
            "energetic and highly engaging voice. Speak noticeably faster than "
            "normal conversation while preserving clean pronunciation and "
            "strong forward momentum."
        ),
    },
}

DELIVERY_PRESETS = {
    "Natural / keep cloned delivery": "",
    "Warm & conversational": (
        "warm, natural, conversational delivery with subtle human pauses"
    ),
    "Calm & serious": (
        "calm, serious, controlled delivery with clear articulation"
    ),
    "Friendly": (
        "friendly, approachable, relaxed conversational delivery"
    ),
    "Professional / formal": (
        "professional, formal, polished delivery with precise articulation"
    ),
    "Documentary": (
        "deep, restrained documentary narration with subtle suspense"
    ),
    "Movie recap": (
        "fast-paced engaging movie recap delivery with strong momentum "
        "and clean articulation"
    ),
    "Excited": (
        "energetic and excited but controlled, clear and never shouting"
    ),
    "Soft / emotional": (
        "soft, intimate and sincere delivery with gentle emotional variation"
    ),
}

# ------------------------------------------------------------
# RESTORE SAVED LIBRARY FROM AN ATTACHED KAGGLE DATASET
# ------------------------------------------------------------

def restore_attached_library_if_needed():
    if META_FILE.exists():
        return None

    candidates = glob.glob(
        "/kaggle/input/**/library.json",
        recursive=True,
    )

    for meta in candidates:
        src_root = Path(meta).parent
        src_voice_dir = src_root / "voices"

        if src_voice_dir.exists():
            shutil.copytree(
                src_root,
                LIBRARY_ROOT,
                dirs_exist_ok=True,
            )
            return str(src_root)

    return None


restored_from = restore_attached_library_if_needed()

# ------------------------------------------------------------
# VOICE LIBRARY
# ------------------------------------------------------------

def empty_library():
    return {
        "version": 6,
        "active_voice_id": None,
        "voices": {},
    }


def load_library():
    if not META_FILE.exists():
        return empty_library()

    try:
        data = json.loads(
            META_FILE.read_text(encoding="utf-8")
        )
        data.setdefault("version", 6)
        data.setdefault("active_voice_id", None)
        data.setdefault("voices", {})
        return data
    except Exception:
        return empty_library()


library = load_library()


def save_library():
    with LIBRARY_LOCK:
        META_FILE.write_text(
            json.dumps(
                library,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )


def voice_path(voice_id):
    return (
        LIBRARY_ROOT
        / library["voices"][voice_id]["file"]
    )


def valid_voice_ids():
    result = []

    for vid, entry in library["voices"].items():
        p = LIBRARY_ROOT / entry.get("file", "")

        if p.exists():
            result.append(vid)

    return result

# ------------------------------------------------------------
# BURMESE-SAFE TEXT CHUNKING
# ------------------------------------------------------------

BOUNDARIES = [
    "။",
    "!",
    "?",
    "…",
    ".",
    "၊",
    ",",
    ";",
    ":",
    "\n",
    " ",
]


def sanitize_tts_text(text):
    """
    Clean invisible/control characters without running WeText normalization.
    """
    text = unicodedata.normalize(
        "NFC",
        str(text or ""),
    )

    # Remove BOM / zero-width characters that can break text parsing.
    text = (
        text
        .replace("\ufeff", "")
        .replace("\u200b", "")
        .replace("\u200c", "")
        .replace("\u200d", "")
        .replace("\u2060", "")
    )

    # Convert most control chars to spaces, but preserve newlines first.
    text = re.sub(
        r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]",
        " ",
        text,
    )

    text = re.sub(
        r"[ \t]+",
        " ",
        text,
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text,
    )

    return text.strip()


def has_speakable_content(text):
    """
    Myanmar base letters are Unicode Letter categories.
    We also permit numbers.
    """
    for ch in str(text or ""):
        category = unicodedata.category(ch)

        if category.startswith("L"):
            return True

        if category.startswith("N"):
            return True

    return False


def split_long_text(text, max_chars=220):
    text = sanitize_tts_text(text)

    if not text:
        return []

    chunks = []
    remaining = text

    while len(remaining) > int(max_chars):
        window = remaining[: int(max_chars) + 1]

        best = -1

        for mark in BOUNDARIES:
            pos = window.rfind(mark)

            if pos > best:
                best = pos

        # Avoid a tiny chunk caused by very old punctuation.
        if best < int(max_chars * 0.55):
            best = int(max_chars)
        else:
            best += 1

        piece = remaining[:best].strip()

        if piece:
            chunks.append(piece)

        remaining = remaining[best:].strip()

    if remaining:
        chunks.append(remaining)

    return chunks


def clean_chunks_for_tts(chunks):
    """
    Remove empty/punctuation-only fragments and merge punctuation-only
    fragments into the previous chunk when useful.
    """
    cleaned = []

    for chunk in chunks:
        chunk = sanitize_tts_text(chunk)

        if not chunk:
            continue

        if not has_speakable_content(chunk):
            if cleaned:
                cleaned[-1] = (
                    cleaned[-1] + " " + chunk
                ).strip()
            continue

        cleaned.append(chunk)

    return cleaned

# ------------------------------------------------------------
# AUDIO HELPERS
# ------------------------------------------------------------

def _mono_float32(wav):
    wav = np.asarray(
        wav,
        dtype=np.float32,
    )

    if wav.ndim == 1:
        return wav

    if wav.ndim == 2:
        return wav.mean(
            axis=1,
            dtype=np.float32,
        )

    return wav.reshape(-1).astype(
        np.float32
    )


# Alias so either helper name works.
mono_float32 = _mono_float32


def trim_edge_silence(
    wav,
    sr,
    threshold_db=-42.0,
    keep_ms=25,
):
    """
    Trim only leading/trailing silence.
    Interior pauses remain untouched.
    """
    wav = _mono_float32(wav)

    if len(wav) == 0:
        return wav

    peak = float(
        np.max(
            np.abs(wav)
        )
    )

    if peak < 1e-7:
        return wav

    threshold = (
        peak
        * (10.0 ** (float(threshold_db) / 20.0))
    )

    active = np.flatnonzero(
        np.abs(wav) >= threshold
    )

    if active.size == 0:
        return wav

    keep = int(
        sr
        * float(keep_ms)
        / 1000.0
    )

    start = max(
        0,
        int(active[0]) - keep,
    )

    end = min(
        len(wav),
        int(active[-1]) + keep + 1,
    )

    return wav[start:end]


def ffmpeg_atempo(
    input_path,
    output_path,
    speed,
):
    """
    Pitch-preserving speed adjustment.
    UI range stays inside FFmpeg atempo's direct 0.5-2.0 range.
    """
    speed = float(speed)

    if abs(speed - 1.0) < 0.001:
        shutil.copy2(
            input_path,
            output_path,
        )
        return str(output_path)

    cmd = [
        "ffmpeg",
        "-y",
        "-loglevel",
        "error",
        "-i",
        str(input_path),
        "-filter:a",
        f"atempo={speed:.4f}",
        "-c:a",
        "pcm_s16le",
        str(output_path),
    ]

    subprocess.run(
        cmd,
        check=True,
    )

    return str(output_path)


def prepare_stream_chunk(
    raw_path,
    output_path,
    speed=1.0,
    trim_edges=True,
    trim_db=-42.0,
):
    """
    Process one completed VoxCPM chunk exactly once.
    The same processed chunk is used for:
      • individual preview
      • continuous stream queue
      • final combined WAV
    """
    raw_path = Path(raw_path)
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    working = raw_path
    temp_speed = None

    try:
        if abs(float(speed) - 1.0) >= 0.001:
            temp_speed = output_path.with_name(
                output_path.stem
                + "_speedtmp.wav"
            )

            ffmpeg_atempo(
                str(raw_path),
                str(temp_speed),
                float(speed),
            )

            working = temp_speed

        wav, sr = sf.read(
            str(working),
            dtype="float32",
            always_2d=False,
        )

        wav = _mono_float32(wav)

        if bool(trim_edges):
            wav = trim_edge_silence(
                wav,
                sr,
                threshold_db=float(trim_db),
                keep_ms=25,
            )

        sf.write(
            str(output_path),
            wav,
            int(sr),
            subtype="PCM_16",
        )

        return (
            str(output_path),
            int(sr),
            int(len(wav)),
        )

    finally:
        if temp_speed is not None:
            try:
                Path(temp_speed).unlink(
                    missing_ok=True
                )
            except Exception:
                pass


def concatenate_prepared_chunks(
    prepared_paths,
    output_path,
    pause_ms=120,
):
    arrays = []
    target_sr = None

    for i, path in enumerate(
        prepared_paths
    ):
        wav, sr = sf.read(
            str(path),
            dtype="float32",
            always_2d=False,
        )

        wav = _mono_float32(wav)
        sr = int(sr)

        if target_sr is None:
            target_sr = sr

        elif sr != target_sr:
            raise RuntimeError(
                f"Sample-rate mismatch: "
                f"{sr} != {target_sr}"
            )

        arrays.append(wav)

        if (
            i != len(prepared_paths) - 1
            and int(pause_ms) > 0
        ):
            arrays.append(
                np.zeros(
                    int(
                        target_sr
                        * int(pause_ms)
                        / 1000.0
                    ),
                    dtype=np.float32,
                )
            )

    if not arrays:
        raise RuntimeError(
            "No prepared chunks "
            "were available."
        )

    final = np.concatenate(arrays)

    sf.write(
        str(output_path),
        final,
        target_sr or 48000,
        subtype="PCM_16",
    )

    return (
        str(output_path),
        target_sr or 48000,
        int(len(final)),
    )


def make_stream_segment_mp3(
    prepared_wav,
    output_mp3,
    pause_ms,
    add_pause=True,
):
    """
    Make a small 64-kbps MP3 browser segment.

    We append the requested join pause to every non-final stream segment.
    The final downloadable file is still WAV.
    """
    prepared_wav = Path(
        prepared_wav
    )

    output_mp3 = Path(
        output_mp3
    )

    temp_wav = (
        output_mp3
        .with_suffix(
            ".segment.wav"
        )
    )

    wav, sr = sf.read(
        str(prepared_wav),
        dtype="float32",
        always_2d=False,
    )

    wav = _mono_float32(wav)
    sr = int(sr)

    if (
        add_pause
        and int(pause_ms) > 0
    ):
        silence = np.zeros(
            int(
                sr
                * int(pause_ms)
                / 1000.0
            ),
            dtype=np.float32,
        )

        wav = np.concatenate(
            [
                wav,
                silence,
            ]
        )

    sf.write(
        str(temp_wav),
        wav,
        sr,
        subtype="PCM_16",
    )

    try:
        cmd = [
            "ffmpeg",
            "-y",
            "-loglevel",
            "error",
            "-i",
            str(temp_wav),
            "-vn",
            "-ac",
            "1",
            "-codec:a",
            "libmp3lame",
            "-b:a",
            "64k",
            str(output_mp3),
        ]

        subprocess.run(
            cmd,
            check=True,
        )

    finally:
        try:
            temp_wav.unlink(
                missing_ok=True
            )
        except Exception:
            pass

    return str(output_mp3)


def file_as_data_url(
    path,
    mime="audio/mpeg",
):
    encoded = base64.b64encode(
        Path(path).read_bytes()
    ).decode("ascii")

    return (
        f"data:{mime};base64,"
        f"{encoded}"
    )


def normalize_media_to_wav(
    source_path,
    output_path,
    start_seconds=0.0,
    max_seconds=20.0,
):
    cmd = [
        "ffmpeg",
        "-y",
        "-loglevel",
        "error",
    ]

    if float(start_seconds) > 0:
        cmd += [
            "-ss",
            str(float(start_seconds)),
        ]

    cmd += [
        "-i",
        str(source_path),
    ]

    if float(max_seconds) > 0:
        cmd += [
            "-t",
            str(float(max_seconds)),
        ]

    cmd += [
        "-vn",
        "-ac",
        "1",
        "-ar",
        "48000",
        "-c:a",
        "pcm_s16le",
        str(output_path),
    ]

    subprocess.run(
        cmd,
        check=True,
    )

    return str(output_path)


def reduce_residual_noise(
    input_path,
    output_path,
    amount=0.70,
):
    import noisereduce as nr

    wav, sr = sf.read(
        str(input_path),
        dtype="float32",
        always_2d=False,
    )

    wav = _mono_float32(wav)

    cleaned = nr.reduce_noise(
        y=wav,
        sr=int(sr),
        stationary=False,
        prop_decrease=float(amount),
    )

    sf.write(
        str(output_path),
        cleaned,
        int(sr),
        subtype="PCM_16",
    )

    return str(output_path)


def extract_vocals_demucs(
    input_path,
    job_dir,
):
    """
    Runs Demucs on CPU so both T4s remain available for VoxCPM2.
    """
    out_root = (
        Path(job_dir)
        / "demucs"
    )

    cmd = [
        sys.executable,
        "-m",
        "demucs.separate",
        "-d",
        "cpu",
        "-n",
        "htdemucs",
        "--two-stems=vocals",
        "--float32",
        "--shifts",
        "0",
        "--overlap",
        "0.10",
        "-o",
        str(out_root),
        str(input_path),
    ]

    subprocess.run(
        cmd,
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    candidates = list(
        out_root.glob(
            "**/vocals.wav"
        )
    )

    if not candidates:
        raise RuntimeError(
            "Demucs finished but "
            "vocals.wav was not found."
        )

    return str(candidates[0])


# ------------------------------------------------------------
# WHISPER LARGE-V3 GPU TRANSCRIPTION
# ------------------------------------------------------------

# GPU lifecycle:
#   1) download/cache ASR assets while both VoxCPM workers remain loaded
#   2) temporarily release physical GPU 1 from VoxCPM2
#   3) run ASR in a separate GPU subprocess
#   4) exit the ASR subprocess -> guaranteed VRAM release
#   5) restore VoxCPM2 on GPU 1
#
# Burmese:
#   TRUE PEFT/LoRA adapter only.
#   The code checks for adapter_config.json + adapter_model.* BEFORE
#   touching GPU 1. A full 6GB Whisper fine-tune is rejected immediately.
#
# Other languages:
#   standard faster-whisper large-v3.

WHISPER_MODEL_NAME = "large-v3"
WHISPER_SAMPLE_RATE = 16000
WHISPER_GPU_ID = 1

ASR_CACHE_ROOT = Path(
    "/kaggle/working/asr_model_cache"
)
ASR_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


def _gb(value):
    return float(value) / (1024.0 ** 3)


def working_disk_free_gb():
    usage = shutil.disk_usage(
        "/kaggle/working"
    )
    return _gb(
        usage.free
    )


def _remove_tree_quiet(path):
    path = Path(path)

    if not path.exists():
        return 0

    before = working_disk_free_gb()

    try:
        shutil.rmtree(
            path,
            ignore_errors=True,
        )
    except Exception:
        pass

    after = working_disk_free_gb()

    return max(
        0.0,
        after - before,
    )


def cleanup_clone_temp_storage(
    preserve_dir=None,
    status_callback=None,
):
    """
    Remove only rebuildable temporary data.

    Permanent voice-library WAVs and final generated voiceovers are NOT
    deleted.
    """
    preserve = (
        Path(preserve_dir).resolve()
        if preserve_dir
        else None
    )

    before = working_disk_free_gb()

    if status_callback is not None:
        status_callback(
            f"Cleaning temporary clone/generation files "
            f"• {before:.1f} GB free…"
        )

    # Long-generation batch folders contain raw chunks + stream MP3s.
    # Final combined WAV files live directly in GENERATED_DIR and remain.
    for path in GENERATED_DIR.glob(
        "batch_*"
    ):
        try:
            shutil.rmtree(
                path,
                ignore_errors=True,
            )
        except Exception:
            pass

    # Keep only the current clone job.
    for path in IMPORT_DIR.glob(
        "clone_*"
    ):
        try:
            if (
                preserve is not None
                and path.resolve() == preserve
            ):
                continue
        except Exception:
            pass

        try:
            shutil.rmtree(
                path,
                ignore_errors=True,
            )
        except Exception:
            pass

    # These are obsolete duplicate-cache layouts created by older Cell 4
    # revisions that used snapshot_download(local_dir=...). v15 uses one
    # normal HF cache instead.
    for path in (
        ASR_CACHE_ROOT / "burmese_full",
        ASR_CACHE_ROOT / "bases",
        Path(
            "/kaggle/working/"
            "whisper_large_v3_cache"
        ),
    ):
        _remove_tree_quiet(
            path
        )

    # hf_xet reconstruction cache is safe to remove. v15 disables Xet.
    for path in (
        Path(
            "/root/.cache/"
            "huggingface/xet"
        ),
        Path(
            "/kaggle/working/"
            ".cache/huggingface/xet"
        ),
        HF_CACHE_ROOT / "xet",
    ):
        _remove_tree_quiet(
            path
        )

    cleanup_incomplete_hf_downloads(
        status_callback=status_callback,
    )

    after = working_disk_free_gb()

    if status_callback is not None:
        status_callback(
            f"Temporary storage cleaned "
            f"• {after:.1f} GB free."
        )

    return (
        before,
        after,
    )


def cleanup_incomplete_hf_downloads(
    status_callback=None,
):
    """
    Delete only incomplete/lock download artifacts from the dedicated HF cache.
    Completed cached models are preserved.
    """
    root = (
        HF_CACHE_ROOT
        / "hub"
    )

    if not root.exists():
        return 0

    removed = 0
    freed_bytes = 0

    candidates = []

    for pattern in (
        "**/*.incomplete",
        "**/*.lock",
    ):
        candidates.extend(
            root.glob(
                pattern
            )
        )

    for path in candidates:
        try:
            if path.is_file():
                size = path.stat().st_size
                path.unlink(
                    missing_ok=True
                )
                freed_bytes += size
                removed += 1
        except Exception:
            pass

    if (
        status_callback is not None
        and removed
    ):
        status_callback(
            f"Removed {removed} incomplete Hugging Face download files "
            f"({freed_bytes / (1024**3):.2f} GB)."
        )

    return removed


def remove_cached_repo(
    repo_id,
    status_callback=None,
):
    """
    Remove one HF repo cache directory when switching away from a previously
    failed giant checkpoint. This is used only for the known large Burmese
    model when Auto falls back to Compact.
    """
    repo_dir = (
        HF_CACHE_ROOT
        / "hub"
        / (
            "models--"
            + str(repo_id)
            .replace("/", "--")
        )
    )

    if not repo_dir.exists():
        return 0.0

    before = working_disk_free_gb()

    try:
        shutil.rmtree(
            repo_dir,
            ignore_errors=True,
        )
    except Exception:
        return 0.0

    after = working_disk_free_gb()
    freed = max(
        0.0,
        after - before,
    )

    if (
        status_callback is not None
        and freed > 0
    ):
        status_callback(
            f"Removed unused cached {repo_id} files "
            f"• freed about {freed:.1f} GB."
        )

    return freed


def ensure_asr_disk_space(
    required_free_gb=11.0,
    preserve_dir=None,
    status_callback=None,
):
    """
    Whisper large-v3-class checkpoints are several GB.
    Keep enough headroom for model files + download temp data + runtime.
    """
    free = working_disk_free_gb()

    if free < float(
        required_free_gb
    ):
        cleanup_clone_temp_storage(
            preserve_dir=preserve_dir,
            status_callback=(
                status_callback
            ),
        )

        free = working_disk_free_gb()

    if free < float(
        required_free_gb
    ):
        raise RuntimeError(
            "Not enough Kaggle disk space for the Burmese Whisper model. "
            f"Only {free:.1f} GB is free; "
            f"at least {required_free_gb:.0f} GB is required. "
            "Delete old notebook outputs/datasets or start a fresh session."
        )

    if status_callback is not None:
        status_callback(
            f"Disk preflight OK • {free:.1f} GB free."
        )

    return free


# Intentionally blank:
# no clearly verified public Burmese Whisper-large-v3 PEFT adapter is
# hard-coded. Paste your own Hugging Face LoRA repo in the UI.
DEFAULT_BURMESE_WHISPER_LORA = ""

# Compact Burmese ASR candidates.
#
# Primary:
#   thantzinphyo/Whisper-Small-ASR
#   ~244M parameters. Its model card reports:
#     • unseen-test CER 5.74%
#     • unseen-test WER 28.22%
#
# Fallbacks are public Whisper-Small Burmese fine-tunes. They are tried
# automatically if the primary repo is unavailable or renamed.
BURMESE_COMPACT_MODEL_CANDIDATES = [
    "thantzinphyo/Whisper-Small-ASR",
    "chuuhtetnaing/whisper-small-myanmar",
    "myatsu/whisper-small-burmese-v2",
]

DEFAULT_BURMESE_COMPACT_MODEL = (
    BURMESE_COMPACT_MODEL_CANDIDATES[0]
)

# Optional large model. This is much larger (~6.18 GB repo) and is only used
# when the user explicitly selects Large or Auto has generous disk headroom.
DEFAULT_BURMESE_LARGE_MODEL = (
    "chuuhtetnaing/whisper-large-v3-myanmar"
)

# Backward-compatible alias used elsewhere in older code paths.
DEFAULT_BURMESE_WHISPER_FINETUNE = (
    DEFAULT_BURMESE_COMPACT_MODEL
)

WHISPER_LANGUAGE_OPTIONS = [
    ("Auto-detect", "auto"),
    ("Burmese / Myanmar", "my"),
    ("English", "en"),
    ("Thai", "th"),
    ("Chinese", "zh"),
    ("Japanese", "ja"),
    ("Korean", "ko"),
    ("Vietnamese", "vi"),
    ("Indonesian", "id"),
    ("Malay", "ms"),
    ("Hindi", "hi"),
    ("Bengali", "bn"),
    ("Urdu", "ur"),
    ("Tamil", "ta"),
    ("Telugu", "te"),
    ("Marathi", "mr"),
    ("Nepali", "ne"),
    ("Khmer", "km"),
    ("Lao", "lo"),
    ("Sinhala", "si"),
    ("Arabic", "ar"),
    ("Persian", "fa"),
    ("Hebrew", "he"),
    ("Turkish", "tr"),
    ("French", "fr"),
    ("German", "de"),
    ("Spanish", "es"),
    ("Portuguese", "pt"),
    ("Italian", "it"),
    ("Russian", "ru"),
    ("Ukrainian", "uk"),
    ("Polish", "pl"),
    ("Dutch", "nl"),
    ("Swedish", "sv"),
    ("Norwegian", "no"),
    ("Danish", "da"),
    ("Finnish", "fi"),
    ("Greek", "el"),
    ("Czech", "cs"),
    ("Romanian", "ro"),
    ("Hungarian", "hu"),
    ("Other / custom Whisper language code", "custom"),
]

# PEFT is required only when Burmese LoRA transcription is requested.
# Do NOT install/upgrade ML packages before VoxCPM starts: changing
# transformers/accelerate at Cell-4 startup can destabilize VoxCPM loading.
import importlib.util


def ensure_peft_runtime(
    status_callback=None,
):
    if importlib.util.find_spec(
        "peft"
    ) is not None:
        return

    if status_callback is not None:
        status_callback(
            "Installing PEFT adapter support without changing "
            "the existing Transformers stack…"
        )

    # --no-deps is deliberate: preserve Kaggle/VoxCPM's already-working
    # torch / transformers / accelerate versions.
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-deps",
            "peft==0.17.1",
        ],
        check=True,
    )

    importlib.invalidate_caches()

    if importlib.util.find_spec(
        "peft"
    ) is None:
        raise RuntimeError(
            "PEFT installation finished but the package "
            "still cannot be imported."
        )


def _safe_repo_dir_name(repo_id):
    return re.sub(
        r"[^A-Za-z0-9._-]+",
        "__",
        str(repo_id),
    )


def _run_with_status_heartbeat(
    func,
    status_callback,
    message,
    every_seconds=10,
):
    """
    Run a blocking download in a helper thread while keeping the UI status
    alive, so a first-time multi-GB base-model download never looks frozen.
    """
    box = {
        "done": False,
        "value": None,
        "error": None,
    }

    def target():
        try:
            box["value"] = func()
        except Exception as e:
            box["error"] = e
        finally:
            box["done"] = True

    t = threading.Thread(
        target=target,
        daemon=True,
    )
    t.start()

    started = time.time()

    while not box["done"]:
        elapsed = int(
            time.time()
            - started
        )

        if status_callback is not None:
            status_callback(
                f"{message} • {elapsed}s elapsed"
            )

        t.join(
            timeout=float(
                every_seconds
            )
        )

    if box["error"] is not None:
        raise box["error"]

    return box["value"]


def prepare_burmese_lora_assets(
    repo_id,
    status_callback=None,
):
    """
    Validate/cache a REAL PEFT LoRA adapter plus its Whisper base.

    v15 does NOT use snapshot_download(local_dir=...), because that can
    create a second multi-GB copy. The returned paths point directly into
    Hugging Face's single cache.
    """
    repo_id = (
        repo_id
        or ""
    ).strip()

    if not repo_id:
        raise ValueError(
            "LoRA mode is selected. Enter a Burmese Whisper "
            "large-v3 PEFT adapter repo."
        )

    from huggingface_hub import (
        HfApi,
        snapshot_download,
    )

    if status_callback is not None:
        status_callback(
            "Checking Burmese Hugging Face repo for a PEFT/LoRA adapter…"
        )

    api = HfApi()

    files = set(
        api.list_repo_files(
            repo_id=repo_id,
            repo_type="model",
        )
    )

    has_config = (
        "adapter_config.json"
        in files
    )

    has_weights = any(
        name in files
        for name in (
            "adapter_model.safetensors",
            "adapter_model.bin",
        )
    )

    if not (
        has_config
        and has_weights
    ):
        raise ValueError(
            f"{repo_id} is not a PEFT/LoRA adapter repo. "
            "It must contain adapter_config.json plus adapter_model weights."
        )

    ensure_peft_runtime(
        status_callback=(
            status_callback
        )
    )

    ensure_asr_disk_space(
        required_free_gb=11.0,
        status_callback=(
            status_callback
        ),
    )

    def download_adapter():
        return snapshot_download(
            repo_id=repo_id,
            repo_type="model",
            cache_dir=str(
                HF_CACHE_ROOT / "hub"
            ),
            local_files_only=False,
        )

    adapter_dir = _run_with_status_heartbeat(
        download_adapter,
        status_callback,
        "Downloading/caching Burmese Whisper LoRA adapter",
        every_seconds=5,
    )

    adapter_config = json.loads(
        (
            Path(adapter_dir)
            / "adapter_config.json"
        ).read_text(
            encoding="utf-8"
        )
    )

    base_model_id = (
        adapter_config.get(
            "base_model_name_or_path"
        )
        or "openai/whisper-large-v3"
    )

    if "whisper" not in base_model_id.lower():
        raise ValueError(
            "The selected adapter's base model is not Whisper: "
            f"{base_model_id}"
        )

    def download_base():
        return snapshot_download(
            repo_id=base_model_id,
            repo_type="model",
            cache_dir=str(
                HF_CACHE_ROOT / "hub"
            ),
            allow_patterns=[
                "*.json",
                "*.safetensors",
                "*.bin",
                "*.txt",
                "*.model",
                "merges.txt",
                "vocab.json",
            ],
            local_files_only=False,
        )

    base_dir = _run_with_status_heartbeat(
        download_base,
        status_callback,
        (
            "Caching Whisper large-v3 base once "
            "(no duplicate local model copy)"
        ),
        every_seconds=10,
    )

    return (
        str(adapter_dir),
        str(base_dir),
        str(base_model_id),
    )


def resolve_public_burmese_compact_repo(
    status_callback=None,
):
    """
    Return the first accessible public compact Burmese ASR repo.

    Built-in public models deliberately use token=False. This avoids a common
    Kaggle failure mode where an expired/incorrect HF_TOKEN is automatically
    attached to a public request and Hugging Face responds with HTTP 401.
    """
    from huggingface_hub import HfApi

    api = HfApi(
        token=False
    )

    failures = []

    for repo_id in BURMESE_COMPACT_MODEL_CANDIDATES:
        if status_callback is not None:
            status_callback(
                f"Checking public Burmese ASR model: {repo_id}"
            )

        try:
            info = api.model_info(
                repo_id=repo_id,
                token=False,
            )

            # Make sure this is actually a model repo with files.
            if info is not None:
                if status_callback is not None:
                    status_callback(
                        f"✓ Using public Burmese ASR model: {repo_id}"
                    )

                return repo_id

        except Exception as e:
            failures.append(
                f"{repo_id}: {type(e).__name__}: {e}"
            )

    raise RuntimeError(
        "None of the built-in public Burmese Whisper-Small models "
        "could be reached. Tried:\\n- "
        + "\\n- ".join(failures)
        + "\\n\\nIf Kaggle has an HF_TOKEN secret configured, remove or fix "
          "the invalid token. Built-in public models do not require a token."
    )


def prepare_burmese_finetune_assets(
    repo_id,
    status_callback=None,
    required_free_gb=3.0,
    model_label="Burmese Whisper model",
    public_repo=True,
):
    """
    Cache a Burmese Whisper fine-tune exactly once.

    For built-in public models we explicitly use token=False. This fixes
    HTTP 401 errors caused by a stale/invalid Hugging Face token in Kaggle.
    """
    repo_id = (
        repo_id
        or DEFAULT_BURMESE_COMPACT_MODEL
    ).strip()

    from huggingface_hub import (
        HfApi,
        snapshot_download,
    )

    hf_token = (
        False
        if public_repo
        else None
    )

    if status_callback is not None:
        status_callback(
            f"Checking {model_label}: {repo_id}"
        )

    api = HfApi(
        token=hf_token
    )

    try:
        info = api.model_info(
            repo_id=repo_id,
            token=hf_token,
        )
    except Exception as e:
        raise RuntimeError(
            f"Cannot access Hugging Face model '{repo_id}'. "
            f"{type(e).__name__}: {e}"
        ) from e

    siblings = getattr(
        info,
        "siblings",
        None,
    ) or []

    files = set()

    for item in siblings:
        name = getattr(
            item,
            "rfilename",
            None,
        )

        if name:
            files.add(
                name
            )

    # Fallback if the model_info response did not include siblings.
    if not files:
        try:
            files = set(
                api.list_repo_files(
                    repo_id=repo_id,
                    repo_type="model",
                    token=hf_token,
                )
            )
        except Exception as e:
            raise RuntimeError(
                f"Could not list files for '{repo_id}'. "
                f"{type(e).__name__}: {e}"
            ) from e

    has_config = (
        "config.json"
        in files
    )

    has_weights = any(
        (
            name.endswith(".safetensors")
            or name.endswith(".bin")
        )
        and (
            "model"
            in Path(name).name
        )
        for name in files
    )

    if not (
        has_config
        and has_weights
    ):
        raise ValueError(
            f"{repo_id} does not look like a complete Whisper "
            "fine-tuned checkpoint."
        )

    cleanup_incomplete_hf_downloads(
        status_callback=status_callback,
    )

    ensure_asr_disk_space(
        required_free_gb=float(
            required_free_gb
        ),
        status_callback=(
            status_callback
        ),
    )

    common_download_kwargs = {
        "repo_id": repo_id,
        "repo_type": "model",
        "cache_dir": str(
            HF_CACHE_ROOT / "hub"
        ),
        "allow_patterns": [
            "*.json",
            "*.safetensors",
            "*.bin",
            "*.txt",
            "*.model",
            "merges.txt",
            "vocab.json",
        ],
        "token": hf_token,
    }

    # Reuse an existing completed cache first.
    try:
        cached = snapshot_download(
            **common_download_kwargs,
            local_files_only=True,
        )

        if status_callback is not None:
            status_callback(
                f"✓ {model_label} found in local cache."
            )

        return str(
            cached
        )

    except Exception:
        pass

    def download_model():
        return snapshot_download(
            **common_download_kwargs,
            local_files_only=False,
        )

    model_dir = _run_with_status_heartbeat(
        download_model,
        status_callback,
        f"Downloading {model_label}",
        every_seconds=10,
    )

    if status_callback is not None:
        status_callback(
            f"✓ {model_label} cached "
            f"• {working_disk_free_gb():.1f} GB free."
        )

    return str(
        model_dir
    )


WHISPER_GPU_WORKER_FILE = Path(
    "/kaggle/working/whisper_gpu_worker.py"
)

WHISPER_GPU_WORKER_SOURCE = r"""
import os
import subprocess
import traceback


def _decode_ffmpeg_float32(
    wav_path,
    sample_rate=16000,
):
    import numpy as np

    cmd = [
        "ffmpeg",
        "-nostdin",
        "-loglevel",
        "error",
        "-i",
        str(wav_path),
        "-vn",
        "-ac",
        "1",
        "-ar",
        str(int(sample_rate)),
        "-f",
        "f32le",
        "pipe:1",
    ]

    proc = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if proc.returncode != 0:
        error_text = proc.stderr.decode(
            "utf-8",
            errors="replace",
        ).strip()

        raise RuntimeError(
            "FFmpeg could not decode the reference audio."
            + (
                "\\n" + error_text
                if error_text
                else ""
            )
        )

    audio = np.frombuffer(
        proc.stdout,
        dtype="<f4",
    ).astype(
        np.float32,
        copy=True,
    )

    if audio.size == 0:
        raise RuntimeError(
            "The decoded reference audio is empty."
        )

    return np.nan_to_num(
        audio,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )


def _transcribe_burmese_full(
    audio,
    model_path,
    result_q,
    physical_gpu_id,
):
    import math
    import unicodedata
    import torch

    from transformers import (
        WhisperProcessor,
        WhisperForConditionalGeneration,
    )

    result_q.put({
        "status": "loading_burmese_full",
        "gpu": physical_gpu_id,
    })

    # Use the exact Burmese language code recommended by the model cards.
    processor = WhisperProcessor.from_pretrained(
        model_path,
        local_files_only=True,
        language="my",
        task="transcribe",
    )

    model = WhisperForConditionalGeneration.from_pretrained(
        model_path,
        local_files_only=True,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )

    device = torch.device(
        "cuda:0"
    )

    model = model.to(
        device
    )

    model.eval()

    # Whisper's decoder context is 448 positions. The language/task/start
    # prompt consumes a few positions, so max_new_tokens=448 is invalid.
    #
    # 320 leaves generous safety headroom and is more than enough for a
    # ~20-28 second speech window.
    max_positions = int(
        getattr(
            model.config,
            "max_target_positions",
            448,
        )
    )

    safe_max_new_tokens = max(
        64,
        min(
            320,
            max_positions - 16,
        ),
    )

    # Whisper's acoustic window is about 30 seconds. The clone UI can accept
    # longer clips, so transcribe them as ordered <=28 s windows instead of
    # allowing the processor/model to truncate the tail.
    sample_rate = 16000
    chunk_seconds = 28.0
    chunk_samples = int(
        sample_rate
        * chunk_seconds
    )

    total_samples = int(
        len(audio)
    )

    total_parts = max(
        1,
        int(
            math.ceil(
                total_samples
                / chunk_samples
            )
        ),
    )

    transcript_parts = []

    for part_index in range(
        total_parts
    ):
        left = (
            part_index
            * chunk_samples
        )

        right = min(
            total_samples,
            left
            + chunk_samples,
        )

        part_audio = audio[
            left:right
        ]

        if part_audio.size == 0:
            continue

        result_q.put({
            "status": "transcribing_burmese_full",
            "gpu": physical_gpu_id,
            "part": part_index + 1,
            "parts": total_parts,
        })

        inputs = processor(
            part_audio,
            sampling_rate=sample_rate,
            return_tensors="pt",
        )

        input_features = (
            inputs.input_features
            .to(
                device=device,
                dtype=torch.float16,
            )
        )

        with torch.inference_mode():
            generated_ids = model.generate(
                input_features,
                language="my",
                task="transcribe",
                do_sample=False,

                # Beam search improves deterministic transcription quality.
                num_beams=5,

                # IMPORTANT:
                # Keep special/start tokens + generated tokens below the
                # model's max_target_positions.
                max_new_tokens=(
                    safe_max_new_tokens
                ),
            )

        part_text = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
        )[0].strip()

        if part_text:
            transcript_parts.append(
                part_text
            )

    transcript = " ".join(
        transcript_parts
    ).strip()

    # The primary compact model was trained/evaluated on normalized Myanmar
    # Unicode, so normalize the decoded string before using it as prompt text.
    transcript = unicodedata.normalize(
        "NFC",
        transcript,
    )

    if not transcript:
        raise RuntimeError(
            "The Burmese Whisper fine-tune returned an empty transcript."
        )

    return {
        "status": "done",
        "gpu": physical_gpu_id,
        "backend": "whisper-burmese-finetune",
        "model": model_path,
        "transcript": transcript,
        "detected": "my",
        "language_probability": None,
    }


def _transcribe_burmese_lora(
    audio,
    adapter_path,
    base_path,
    result_q,
    physical_gpu_id,
):
    import math
    import unicodedata
    import torch

    from peft import PeftModel

    from transformers import (
        WhisperProcessor,
        WhisperForConditionalGeneration,
    )

    result_q.put({
        "status": "loading_burmese_base",
        "gpu": physical_gpu_id,
    })

    processor = WhisperProcessor.from_pretrained(
        base_path,
        language="my",
        task="transcribe",
        local_files_only=True,
    )

    base_model = WhisperForConditionalGeneration.from_pretrained(
        base_path,
        local_files_only=True,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )

    result_q.put({
        "status": "attaching_burmese_lora",
        "gpu": physical_gpu_id,
    })

    model = PeftModel.from_pretrained(
        base_model,
        adapter_path,
        local_files_only=True,
    )

    device = torch.device(
        "cuda:0"
    )

    model = model.to(
        device
    )

    model.eval()

    # Read max_target_positions from the underlying Whisper model when PEFT
    # wraps it.
    whisper_config = getattr(
        model,
        "config",
        None,
    )

    max_positions = int(
        getattr(
            whisper_config,
            "max_target_positions",
            448,
        )
    )

    safe_max_new_tokens = max(
        64,
        min(
            320,
            max_positions - 16,
        ),
    )

    sample_rate = 16000
    chunk_seconds = 28.0
    chunk_samples = int(
        sample_rate
        * chunk_seconds
    )

    total_samples = int(
        len(audio)
    )

    total_parts = max(
        1,
        int(
            math.ceil(
                total_samples
                / chunk_samples
            )
        ),
    )

    transcript_parts = []

    for part_index in range(
        total_parts
    ):
        left = (
            part_index
            * chunk_samples
        )

        right = min(
            total_samples,
            left
            + chunk_samples,
        )

        part_audio = audio[
            left:right
        ]

        if part_audio.size == 0:
            continue

        result_q.put({
            "status": "transcribing_burmese_lora",
            "gpu": physical_gpu_id,
            "part": part_index + 1,
            "parts": total_parts,
        })

        inputs = processor(
            part_audio,
            sampling_rate=sample_rate,
            return_tensors="pt",
        )

        input_features = (
            inputs.input_features
            .to(
                device=device,
                dtype=torch.float16,
            )
        )

        with torch.inference_mode():
            generated_ids = model.generate(
                input_features,
                language="my",
                task="transcribe",
                do_sample=False,
                num_beams=5,
                max_new_tokens=(
                    safe_max_new_tokens
                ),
            )

        part_text = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
        )[0].strip()

        if part_text:
            transcript_parts.append(
                part_text
            )

    transcript = " ".join(
        transcript_parts
    ).strip()

    transcript = unicodedata.normalize(
        "NFC",
        transcript,
    )

    if not transcript:
        raise RuntimeError(
            "The Burmese Whisper LoRA returned an empty transcript."
        )

    return {
        "status": "done",
        "gpu": physical_gpu_id,
        "backend": "whisper-large-v3-peft-lora",
        "model": adapter_path,
        "transcript": transcript,
        "detected": "my",
        "language_probability": None,
    }


def _transcribe_other_whisper(
    audio,
    language,
    cache_dir,
    result_q,
    physical_gpu_id,
):
    from faster_whisper import WhisperModel

    result_q.put({
        "status": "loading_whisper",
        "gpu": physical_gpu_id,
    })

    model = WhisperModel(
        "large-v3",
        device="cuda",
        device_index=0,
        compute_type="float16",
        download_root=str(
            cache_dir
        ),
    )

    result_q.put({
        "status": "transcribing_whisper",
        "gpu": physical_gpu_id,
    })

    lang = (
        None
        if language in (
            None,
            "",
            "auto",
        )
        else str(
            language
        )
    )

    segments, info = model.transcribe(
        audio,
        language=lang,
        task="transcribe",
        beam_size=5,
        vad_filter=True,
        condition_on_previous_text=True,
        word_timestamps=False,
    )

    parts = []

    for seg in segments:
        value = (
            seg.text
            or ""
        ).strip()

        if value:
            parts.append(
                value
            )

    transcript = " ".join(
        parts
    ).strip()

    if not transcript:
        raise RuntimeError(
            "Whisper returned an empty transcript."
        )

    return {
        "status": "done",
        "gpu": physical_gpu_id,
        "backend": "faster-whisper-large-v3",
        "model": "large-v3",
        "transcript": transcript,
        "detected": getattr(
            info,
            "language",
            None,
        ),
        "language_probability": getattr(
            info,
            "language_probability",
            None,
        ),
    }


def whisper_gpu_main(
    physical_gpu_id,
    wav_path,
    language,
    cache_dir,
    burmese_mode,
    burmese_model_path,
    burmese_base_path,
    result_q,
):
    os.environ[
        "CUDA_VISIBLE_DEVICES"
    ] = str(
        physical_gpu_id
    )

    try:
        result_q.put({
            "status": "decoding",
            "gpu": physical_gpu_id,
        })

        audio = _decode_ffmpeg_float32(
            wav_path,
            sample_rate=16000,
        )

        if language == "my":
            if burmese_mode == "lora":
                result = _transcribe_burmese_lora(
                    audio=audio,
                    adapter_path=burmese_model_path,
                    base_path=burmese_base_path,
                    result_q=result_q,
                    physical_gpu_id=physical_gpu_id,
                )

            else:
                result = _transcribe_burmese_full(
                    audio=audio,
                    model_path=burmese_model_path,
                    result_q=result_q,
                    physical_gpu_id=physical_gpu_id,
                )

        else:
            result = _transcribe_other_whisper(
                audio=audio,
                language=language,
                cache_dir=cache_dir,
                result_q=result_q,
                physical_gpu_id=physical_gpu_id,
            )

        result_q.put(
            result
        )

    except Exception as e:
        result_q.put({
            "status": "error",
            "gpu": physical_gpu_id,
            "error": repr(
                e
            ),
            "traceback": traceback.format_exc(),
        })
"""

WHISPER_GPU_WORKER_FILE.write_text(
    WHISPER_GPU_WORKER_SOURCE,
    encoding="utf-8",
)

compile(
    WHISPER_GPU_WORKER_SOURCE,
    str(WHISPER_GPU_WORKER_FILE),
    "exec",
)

if "whisper_gpu_worker" in sys.modules:
    del sys.modules["whisper_gpu_worker"]

from whisper_gpu_worker import whisper_gpu_main


def _release_voxcpm_gpu_worker(
    gpu_id=WHISPER_GPU_ID,
):
    global _VOX_SERVICE

    idx = _VOX_SERVICE.gpu_ids.index(
        gpu_id
    )

    proc = _VOX_SERVICE.procs[
        idx
    ]

    if proc is None:
        return

    if proc.is_alive():
        proc.terminate()
        proc.join(
            timeout=20
        )

    if proc.is_alive():
        try:
            proc.kill()
        except Exception:
            pass

        proc.join(
            timeout=10
        )

    _VOX_SERVICE.procs[
        idx
    ] = None

    time.sleep(
        1.5
    )


def _restore_voxcpm_gpu_worker(
    gpu_id=WHISPER_GPU_ID,
    timeout=WORKER_STARTUP_TIMEOUT,
    status_callback=None,
):
    global _VOX_SERVICE

    idx = _VOX_SERVICE.gpu_ids.index(
        gpu_id
    )

    old = _VOX_SERVICE.procs[
        idx
    ]

    if (
        old is not None
        and old.is_alive()
    ):
        return

    if status_callback is not None:
        status_callback(
            f"Reloading VoxCPM2 on GPU {gpu_id}…"
        )

    p = _VOX_SERVICE.ctx.Process(
        target=worker_main,
        args=(
            gpu_id,
            _VOX_SERVICE.model_dir,
            _VOX_SERVICE.task_q,
            _VOX_SERVICE.result_q,
            _VOX_SERVICE.optimize,
        ),
        name=f"VoxCPM2-GPU-{gpu_id}",
    )

    p.start()

    _VOX_SERVICE.procs[
        idx
    ] = p

    deadline = (
        time.time()
        + float(timeout)
    )

    while True:
        if p.exitcode is not None:
            raise RuntimeError(
                f"VoxCPM2 GPU {gpu_id} worker exited while restoring "
                f"(exit code {p.exitcode})."
            )

        if time.time() > deadline:
            try:
                p.terminate()
            except Exception:
                pass

            raise TimeoutError(
                f"Timed out reloading VoxCPM2 on GPU {gpu_id}."
            )

        try:
            msg = _VOX_SERVICE.result_q.get(
                timeout=5
            )
        except pyqueue.Empty:
            continue

        if msg.get(
            "gpu"
        ) != gpu_id:
            continue

        status = msg.get(
            "status"
        )

        if status == "loading":
            if status_callback is not None:
                status_callback(
                    f"VoxCPM2 GPU {gpu_id} is loading…"
                )

        elif status == "ready":
            _VOX_SERVICE.sample_rate = msg.get(
                "sample_rate",
                _VOX_SERVICE.sample_rate,
            )

            if status_callback is not None:
                status_callback(
                    f"VoxCPM2 GPU {gpu_id} restored."
                )

            return

        elif status == "startup_error":
            raise RuntimeError(
                f"VoxCPM2 GPU {gpu_id} failed to restore:\n"
                f"{msg.get('error')}\n\n"
                f"{msg.get('traceback', '')}"
            )


def transcribe_reference(
    wav_path,
    language="my",
    burmese_mode="auto",
    burmese_model_repo="",
    status_callback=None,
    gpu_id=WHISPER_GPU_ID,
):
    """
    Burmese:
      mode="finetune" -> verified full Whisper large-v3 Burmese model
      mode="lora"     -> user-supplied PEFT adapter

    Downloads happen before GPU 1 is released.
    """
    global _VOX_SERVICE

    burmese_model_path = ""
    burmese_base_path = ""
    display_model_name = ""

    if language == "my":
        cleanup_incomplete_hf_downloads(
            status_callback=status_callback,
        )

        requested_mode = (
            burmese_mode
            or "auto"
        )

        if requested_mode == "auto":
            free_gb = working_disk_free_gb()

            # The large-v3 repo is ~6.2 GB. Keep comfortable temporary/runtime
            # headroom; otherwise use the 244M Whisper-Small Burmese model.
            if free_gb >= 12.0:
                requested_mode = "large"

                if status_callback is not None:
                    status_callback(
                        f"Auto Burmese ASR: {free_gb:.1f} GB free → "
                        "using Whisper large-v3 Burmese."
                    )
            else:
                requested_mode = "compact"

                # A failed earlier large-v3 attempt can leave several GB cached.
                # Auto compact mode no longer needs that large repository.
                remove_cached_repo(
                    DEFAULT_BURMESE_LARGE_MODEL,
                    status_callback=status_callback,
                )

                if status_callback is not None:
                    status_callback(
                        f"Auto Burmese ASR: {working_disk_free_gb():.1f} GB free → "
                        "using compact Whisper-Small Burmese."
                    )

        if requested_mode == "lora":
            (
                burmese_model_path,
                burmese_base_path,
                base_model_id,
            ) = prepare_burmese_lora_assets(
                burmese_model_repo,
                status_callback=(
                    status_callback
                ),
            )

            display_model_name = (
                burmese_model_repo
            )

            burmese_mode = "lora"

        elif requested_mode == "large":
            selected_repo = (
                burmese_model_repo
                or DEFAULT_BURMESE_LARGE_MODEL
            )

            burmese_model_path = (
                prepare_burmese_finetune_assets(
                    selected_repo,
                    status_callback=(
                        status_callback
                    ),
                    required_free_gb=10.0,
                    model_label=(
                        "Burmese Whisper large-v3 fine-tune"
                    ),
                    public_repo=True,
                )
            )

            display_model_name = (
                selected_repo
            )

            burmese_mode = "finetune"

        else:
            # Compact high-accuracy Burmese model.
            #
            # If the user did not explicitly override the repo, resolve the
            # first currently accessible public model from our fallback list.
            selected_repo = (
                burmese_model_repo.strip()
                if (
                    burmese_model_repo
                    and burmese_model_repo.strip()
                )
                else resolve_public_burmese_compact_repo(
                    status_callback=(
                        status_callback
                    )
                )
            )

            # Whisper-Small is ~244M parameters and is much easier on both
            # Kaggle disk and T4 VRAM than large-v3.
            burmese_model_path = (
                prepare_burmese_finetune_assets(
                    selected_repo,
                    status_callback=(
                        status_callback
                    ),
                    required_free_gb=2.5,
                    model_label=(
                        "Compact Burmese Whisper-Small fine-tune"
                    ),
                    public_repo=True,
                )
            )

            display_model_name = (
                selected_repo
            )

            burmese_mode = "finetune"

    if status_callback is not None:
        status_callback(
            f"Freeing GPU {gpu_id} from VoxCPM2 for ASR…"
        )

    _release_voxcpm_gpu_worker(
        gpu_id=gpu_id
    )

    ctx = get_context(
        "spawn"
    )

    result_q = ctx.Queue()

    proc = ctx.Process(
        target=whisper_gpu_main,
        args=(
            gpu_id,
            str(
                wav_path
            ),
            language,
            str(
                ASR_CACHE_ROOT
            ),
            burmese_mode,
            burmese_model_path,
            burmese_base_path,
            result_q,
        ),
        name=(
            f"ASR-GPU-{gpu_id}"
        ),
    )

    transcript_result = None
    started = time.time()
    last_heartbeat = 0.0
    current_stage = (
        "Starting GPU ASR"
    )

    try:
        proc.start()

        while True:
            try:
                msg = result_q.get(
                    timeout=2
                )

            except pyqueue.Empty:
                if (
                    proc.exitcode
                    is not None
                    and transcript_result
                    is None
                ):
                    raise RuntimeError(
                        "ASR GPU subprocess exited unexpectedly "
                        f"with code {proc.exitcode}."
                    )

                now = time.time()

                if (
                    status_callback
                    is not None
                    and now
                    - last_heartbeat
                    >= 10
                ):
                    status_callback(
                        f"{current_stage} • "
                        f"{int(now - started)}s elapsed"
                    )

                    last_heartbeat = now

                continue

            status = msg.get(
                "status"
            )

            if status == "decoding":
                current_stage = (
                    "Decoding reference audio"
                )

            elif status == "loading_burmese_full":
                current_stage = (
                    "Loading Burmese Whisper large-v3 fine-tune on GPU 1"
                )

            elif status == "transcribing_burmese_full":
                part = msg.get(
                    "part",
                    1,
                )
                parts = msg.get(
                    "parts",
                    1,
                )

                current_stage = (
                    "Transcribing Burmese with fine-tuned Whisper "
                    f"• part {part}/{parts}"
                )

            elif status == "loading_burmese_base":
                current_stage = (
                    "Loading Whisper large-v3 base on GPU 1"
                )

            elif status == "attaching_burmese_lora":
                current_stage = (
                    "Attaching Burmese PEFT/LoRA adapter"
                )

            elif status == "transcribing_burmese_lora":
                part = msg.get(
                    "part",
                    1,
                )
                parts = msg.get(
                    "parts",
                    1,
                )

                current_stage = (
                    "Transcribing Burmese with Whisper LoRA "
                    f"• part {part}/{parts}"
                )

            elif status == "loading_whisper":
                current_stage = (
                    "Loading Whisper large-v3 on GPU 1"
                )

            elif status == "transcribing_whisper":
                current_stage = (
                    f"Transcribing language={language} on GPU 1"
                )

            elif status == "done":
                transcript_result = (
                    msg.get(
                        "transcript",
                        "",
                    ),
                    msg.get(
                        "detected"
                    ),
                    msg.get(
                        "language_probability"
                    ),
                    msg.get(
                        "backend",
                        "unknown",
                    ),
                    (
                        display_model_name
                        if language
                        == "my"
                        else msg.get(
                            "model",
                            "large-v3",
                        )
                    ),
                )

                break

            elif status == "error":
                raise RuntimeError(
                    "GPU transcription failed:\n"
                    f"{msg.get('error')}\n\n"
                    f"{msg.get('traceback', '')}"
                )

            if (
                status_callback
                is not None
                and status
                not in (
                    "done",
                    "error",
                )
            ):
                status_callback(
                    current_stage
                )

        if transcript_result is None:
            raise RuntimeError(
                "GPU transcription ended without a result."
            )

        return transcript_result

    finally:
        if proc.is_alive():
            proc.terminate()
            proc.join(
                timeout=10
            )

        try:
            result_q.close()
        except Exception:
            pass

        time.sleep(
            1.0
        )

        _restore_voxcpm_gpu_worker(
            gpu_id=gpu_id,
            status_callback=(
                status_callback
            ),
        )


# ------------------------------------------------------------
# UPLOAD / REMOTE MEDIA HELPERS
# ------------------------------------------------------------

def _first_upload(widget):
    value = widget.value

    if not value:
        return None

    if isinstance(
        value,
        (tuple, list),
    ):
        item = value[0]

        return {
            "name": item.get(
                "name",
                "upload.bin",
            ),
            "content": bytes(
                item["content"]
            ),
        }

    if isinstance(
        value,
        dict,
    ):
        first_key = next(
            iter(value)
        )

        item = value[
            first_key
        ]

        name = (
            item
            .get(
                "metadata",
                {},
            )
            .get(
                "name",
                first_key,
            )
        )

        return {
            "name": name,
            "content": bytes(
                item["content"]
            ),
        }

    return None


def save_upload_to_dir(
    upload_widget,
    dest_dir,
    fallback_name="upload.bin",
):
    item = _first_upload(
        upload_widget
    )

    if item is None:
        raise ValueError(
            "No local file "
            "has been uploaded."
        )

    name = Path(
        item["name"]
        or fallback_name
    ).name

    dest = (
        Path(dest_dir)
        / name
    )

    dest.write_bytes(
        item["content"]
    )

    return str(dest)


def save_cookie_upload(
    cookie_widget,
):
    item = _first_upload(
        cookie_widget
    )

    if item is None:
        return None

    path = (
        COOKIE_DIR
        / "cookies.txt"
    )

    path.write_bytes(
        item["content"]
    )

    return str(path)


def download_remote_media(
    url,
    job_dir,
    cookie_file=None,
    user_agent="",
    referer="",
):
    """
    Use only with media the user is permitted to access/download.
    """
    url = (
        url or ""
    ).strip()

    if not url:
        raise ValueError(
            "Remote URL is empty."
        )

    host = (
        urlparse(url)
        .netloc
        .lower()
    )

    job_dir = Path(
        job_dir
    )

    job_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Google Drive
    if (
        "drive.google.com" in host
        or "docs.google.com" in host
    ):
        import gdown

        out = (
            job_dir
            / "drive_source"
        )

        result = gdown.download(
            url=url,
            output=str(out),
            quiet=False,
            fuzzy=True,
        )

        if not result:
            raise RuntimeError(
                "Google Drive "
                "download failed."
            )

        return str(result)

    # MEGA public file link
    if (
        "mega.nz" in host
        or "mega.co.nz" in host
    ):
        from mega import Mega

        mega = Mega()
        client = mega.login()

        before = set(
            job_dir.iterdir()
        )

        result = client.download_url(
            url,
            dest_path=str(
                job_dir
            ),
        )

        if (
            result
            and Path(result).exists()
        ):
            return str(result)

        after = set(
            job_dir.iterdir()
        )

        new_files = [
            p
            for p in (
                after - before
            )
            if p.is_file()
        ]

        if not new_files:
            raise RuntimeError(
                "MEGA download did "
                "not create a file."
            )

        return str(
            max(
                new_files,
                key=lambda p: p.stat().st_size,
            )
        )

    # YouTube / TikTok / direct HTTP / yt-dlp supported sites.
    import yt_dlp

    outtmpl = str(
        job_dir
        / "remote_source.%(ext)s"
    )

    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": outtmpl,
        "noplaylist": True,
        "quiet": True,
        "retries": 3,
        "fragment_retries": 3,
        "postprocessors": [
            {
                "key": "FFmpegExtractAudio",
                "preferredcodec": "wav",
            }
        ],
    }

    if (
        cookie_file
        and Path(
            cookie_file
        ).exists()
    ):
        ydl_opts[
            "cookiefile"
        ] = str(
            cookie_file
        )

    headers = {}

    if user_agent.strip():
        headers[
            "User-Agent"
        ] = (
            user_agent.strip()
        )

    if referer.strip():
        headers[
            "Referer"
        ] = (
            referer.strip()
        )

    if headers:
        ydl_opts[
            "http_headers"
        ] = headers

    with yt_dlp.YoutubeDL(
        ydl_opts
    ) as ydl:
        ydl.download(
            [url]
        )

    candidates = [
        p
        for p in job_dir.glob(
            "remote_source.*"
        )
        if (
            p.is_file()
            and p.suffix != ".part"
        )
    ]

    if not candidates:
        raise RuntimeError(
            "Remote download did "
            "not produce a media file."
        )

    return str(
        max(
            candidates,
            key=lambda p: p.stat().st_size,
        )
    )

# ------------------------------------------------------------
# DUAL-GPU SERVICE
# ------------------------------------------------------------

# Cell 4 writes its own worker module so it cannot accidentally reuse an
# older/stale worker file from a previous notebook revision.
TTS_WORKER_FILE = Path(
    "/kaggle/working/voxcpm_dual_worker_v13.py"
)

TTS_WORKER_SOURCE = r"""
import os
import random
import inspect
import traceback


def worker_main(
    physical_gpu_id,
    model_dir,
    task_q,
    result_q,
    optimize=False,
):
    # Set this before importing torch / VoxCPM.
    os.environ[
        "CUDA_VISIBLE_DEVICES"
    ] = str(
        physical_gpu_id
    )

    # Avoid accidental online lookups while loading a local VoxCPM model.
    # This does not affect later ASR downloads, which happen in other processes.
    os.environ.setdefault(
        "TOKENIZERS_PARALLELISM",
        "false",
    )

    try:
        result_q.put({
            "status": "booting",
            "gpu": physical_gpu_id,
            "stage": "starting child process",
        })

        import numpy as np
        import torch
        import soundfile as sf

        result_q.put({
            "status": "imports_ready",
            "gpu": physical_gpu_id,
            "stage": "torch/soundfile imported",
        })

        from voxcpm import VoxCPM

        result_q.put({
            "status": "loading",
            "gpu": physical_gpu_id,
            "stage": "VoxCPM.from_pretrained",
        })

        model = VoxCPM.from_pretrained(
            model_dir,
            load_denoiser=False,
            optimize=bool(
                optimize
            ),
            device="cuda",
        )

        result_q.put({
            "status": "model_loaded",
            "gpu": physical_gpu_id,
            "stage": "model object loaded on CUDA",
        })

        try:
            sample_rate = int(
                model.tts_model.sample_rate
            )
        except Exception:
            sample_rate = 48000

        # Detect exactly what the installed VoxCPM build accepts.
        try:
            sig = inspect.signature(
                model._generate
            )

            params = sig.parameters

            accepts_any_kwarg = any(
                p.kind
                == inspect.Parameter.VAR_KEYWORD
                for p in params.values()
            )

            supported = set(
                params.keys()
            )

        except Exception:
            accepts_any_kwarg = True
            supported = set()

        result_q.put({
            "status": "ready",
            "gpu": physical_gpu_id,
            "sample_rate": sample_rate,
            "supported_params": sorted(
                supported
            ),
        })

    except Exception as e:
        result_q.put({
            "status": "startup_error",
            "gpu": physical_gpu_id,
            "error": repr(e),
            "traceback": traceback.format_exc(),
        })
        return

    while True:
        task = task_q.get()

        if task is None:
            break

        batch_id = task[
            "batch_id"
        ]

        task_id = task[
            "task_id"
        ]

        chunk_index = task.get(
            "chunk_index",
            0,
        )

        output_path = task[
            "output_path"
        ]

        kwargs = dict(
            task["kwargs"]
        )

        try:
            os.makedirs(
                os.path.dirname(
                    output_path
                ),
                exist_ok=True,
            )

            # Older VoxCPM builds do not accept seed=. If one appears,
            # emulate it through process RNGs and remove the keyword.
            requested_seed = kwargs.get(
                "seed",
                None,
            )

            if (
                requested_seed
                is not None
                and not accepts_any_kwarg
                and "seed"
                not in supported
            ):
                requested_seed = int(
                    requested_seed
                )

                random.seed(
                    requested_seed
                )

                np.random.seed(
                    requested_seed
                    % (2**32 - 1)
                )

                torch.manual_seed(
                    requested_seed
                )

                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(
                        requested_seed
                    )

                kwargs.pop(
                    "seed",
                    None,
                )

            dropped_kwargs = []

            if not accepts_any_kwarg:
                for key in list(
                    kwargs.keys()
                ):
                    if key not in supported:
                        dropped_kwargs.append(
                            key
                        )

                        kwargs.pop(
                            key,
                            None,
                        )

            wav = model.generate(
                **kwargs
            )

            sf.write(
                output_path,
                wav,
                sample_rate,
                subtype="PCM_16",
            )

            result_q.put({
                "status": "ok",
                "batch_id": batch_id,
                "task_id": task_id,
                "chunk_index": chunk_index,
                "gpu": physical_gpu_id,
                "output_path": output_path,
                "sample_rate": sample_rate,
                "samples": int(
                    len(wav)
                ),
                "duration": float(
                    len(wav)
                    / sample_rate
                ),
                "dropped_kwargs": dropped_kwargs,
            })

        except Exception as e:
            result_q.put({
                "status": "error",
                "batch_id": batch_id,
                "task_id": task_id,
                "chunk_index": chunk_index,
                "gpu": physical_gpu_id,
                "error": repr(e),
                "traceback": traceback.format_exc(),
            })
"""

TTS_WORKER_FILE.write_text(
    TTS_WORKER_SOURCE,
    encoding="utf-8",
)

compile(
    TTS_WORKER_SOURCE,
    str(TTS_WORKER_FILE),
    "exec",
)

sys.path.insert(
    0,
    "/kaggle/working",
)

if "voxcpm_dual_worker_v13" in sys.modules:
    del sys.modules[
        "voxcpm_dual_worker_v13"
    ]

from voxcpm_dual_worker_v13 import worker_main


class DualGPUVoxCPM:
    def __init__(
        self,
        model_dir,
        gpu_ids=(0, 1),
        optimize=False,
        startup_timeout=900,
        heartbeat_seconds=10,
    ):
        self.model_dir = model_dir
        self.gpu_ids = tuple(
            gpu_ids
        )
        self.optimize = bool(
            optimize
        )

        self.ctx = get_context(
            "spawn"
        )

        self.task_q = (
            self.ctx.Queue()
        )

        self.result_q = (
            self.ctx.Queue()
        )

        # Fixed-size list matters because ASR temporarily sets GPU 1 to None.
        self.procs = [
            None
            for _ in self.gpu_ids
        ]

        self.sample_rate = 48000
        self.worker_supported_params = {}

        print(
            "Starting VoxCPM2 workers sequentially "
            "to avoid model-load/cache contention..."
        )

        for idx, gpu_id in enumerate(
            self.gpu_ids
        ):
            self._start_worker_and_wait(
                index=idx,
                gpu_id=gpu_id,
                startup_timeout=startup_timeout,
                heartbeat_seconds=heartbeat_seconds,
            )

        print(
            "✓ Both VoxCPM2 workers are ready."
        )

    def _start_worker_and_wait(
        self,
        index,
        gpu_id,
        startup_timeout,
        heartbeat_seconds,
    ):
        print(
            f"\\n→ Starting physical GPU {gpu_id}..."
        )

        p = self.ctx.Process(
            target=worker_main,
            args=(
                gpu_id,
                self.model_dir,
                self.task_q,
                self.result_q,
                self.optimize,
            ),
            name=(
                f"VoxCPM2-GPU-{gpu_id}"
            ),
        )

        p.start()

        self.procs[
            index
        ] = p

        print(
            f"  worker PID {p.pid}"
        )

        deadline = (
            time.time()
            + float(
                startup_timeout
            )
        )

        started = time.time()
        last_heartbeat = 0.0
        last_stage = "child process launched"

        while True:
            now = time.time()

            if p.exitcode is not None:
                raise RuntimeError(
                    f"GPU {gpu_id} worker exited during startup "
                    f"(exit code {p.exitcode}) while at stage: "
                    f"{last_stage}"
                )

            if now > deadline:
                try:
                    p.terminate()
                    p.join(
                        timeout=10
                    )
                except Exception:
                    pass

                raise TimeoutError(
                    f"GPU {gpu_id} timed out after "
                    f"{int(now - started)}s at startup stage: "
                    f"{last_stage}. "
                    "This usually means model loading is blocked, "
                    "not that the GPU is busy."
                )

            try:
                msg = self.result_q.get(
                    timeout=2
                )

            except pyqueue.Empty:
                if (
                    now
                    - last_heartbeat
                    >= heartbeat_seconds
                ):
                    print(
                        f"  … GPU {gpu_id}: {last_stage} "
                        f"• {int(now - started)}s elapsed"
                    )

                    last_heartbeat = now

                continue

            # During sequential startup there should be no other new worker,
            # but ignore any stale queue message from an older batch.
            if msg.get(
                "gpu"
            ) != gpu_id:
                continue

            status = msg.get(
                "status"
            )

            if status == "booting":
                last_stage = msg.get(
                    "stage",
                    "booting",
                )
                print(
                    f"  • GPU {gpu_id}: {last_stage}"
                )

            elif status == "imports_ready":
                last_stage = msg.get(
                    "stage",
                    "imports ready",
                )
                print(
                    f"  • GPU {gpu_id}: {last_stage}"
                )

            elif status == "loading":
                last_stage = msg.get(
                    "stage",
                    "VoxCPM.from_pretrained",
                )
                print(
                    f"  • GPU {gpu_id}: {last_stage}"
                )

            elif status == "model_loaded":
                last_stage = msg.get(
                    "stage",
                    "model loaded",
                )
                print(
                    f"  • GPU {gpu_id}: {last_stage}"
                )

            elif status == "ready":
                self.sample_rate = msg.get(
                    "sample_rate",
                    self.sample_rate,
                )

                self.worker_supported_params[
                    gpu_id
                ] = set(
                    msg.get(
                        "supported_params",
                        [],
                    )
                )

                print(
                    f"✓ GPU {gpu_id} ready "
                    f"({self.sample_rate} Hz) "
                    f"after {int(time.time() - started)}s"
                )

                return

            elif status == "startup_error":
                raise RuntimeError(
                    f"GPU {gpu_id} failed to start:\\n"
                    f"{msg.get('error')}\\n\\n"
                    f"{msg.get('traceback', '')}"
                )

    def run(
        self,
        task_specs,
        on_result=None,
        stop_event=None,
    ):
        """
        Run tasks across both GPU workers, keeping at most one
        in-flight task per GPU.

        Stop prevents new chunks from being scheduled.
        The one or two chunks already running are allowed to finish.

        Returns:
            (sorted_results, stopped)
        """
        if not task_specs:
            return [], False

        batch_id = uuid.uuid4().hex
        total = len(task_specs)
        results = []

        next_index = 0
        in_flight = {}
        stopped = False

        def submit_one(index):
            task_id = (
                f"{batch_id}_{index}"
            )

            payload = dict(
                task_specs[index]
            )

            payload[
                "batch_id"
            ] = batch_id

            payload[
                "task_id"
            ] = task_id

            in_flight[
                task_id
            ] = index

            self.task_q.put(
                payload
            )

        initial = min(
            len(self.gpu_ids),
            total,
        )

        for _ in range(
            initial
        ):
            submit_one(
                next_index
            )
            next_index += 1

        while in_flight:
            if (
                stop_event
                is not None
                and stop_event.is_set()
            ):
                stopped = True

            for gpu_id, proc in zip(
                self.gpu_ids,
                self.procs,
            ):
                if proc is None:
                    raise RuntimeError(
                        f"GPU worker {gpu_id} is temporarily unavailable. "
                        "Wait for voice-clone ASR to finish and VoxCPM2 "
                        "to reload."
                    )

                if proc.exitcode is not None:
                    raise RuntimeError(
                        f"GPU worker {gpu_id} died during generation "
                        f"with exit code {proc.exitcode}."
                    )

            try:
                msg = self.result_q.get(
                    timeout=1
                )

            except pyqueue.Empty:
                continue

            if msg.get(
                "batch_id"
            ) != batch_id:
                continue

            task_id = msg.get(
                "task_id"
            )

            if task_id not in in_flight:
                continue

            idx = in_flight.pop(
                task_id
            )

            if msg.get(
                "status"
            ) != "ok":
                failing_text = ""

                if 0 <= idx < len(
                    task_specs
                ):
                    try:
                        failing_text = (
                            task_specs[idx]
                            .get(
                                "kwargs",
                                {},
                            )
                            .get(
                                "text",
                                "",
                            )
                        )
                    except Exception:
                        failing_text = ""

                raise RuntimeError(
                    f"GPU {msg.get('gpu')} generation failed "
                    f"on chunk {idx + 1}/{total}:\\n"
                    f"{msg.get('error')}\\n"
                    f"Failing text: "
                    f"{repr(str(failing_text)[:240])}\\n\\n"
                    f"{msg.get('traceback', '')}"
                )

            results.append(
                msg
            )

            if on_result is not None:
                try:
                    on_result(
                        msg,
                        len(results),
                        total,
                    )
                except Exception as callback_error:
                    print(
                        "⚠ Preview callback error:",
                        repr(
                            callback_error
                        ),
                    )

            if (
                not stopped
                and next_index < total
            ):
                submit_one(
                    next_index
                )
                next_index += 1

        return (
            sorted(
                results,
                key=lambda r: r.get(
                    "chunk_index",
                    0,
                ),
            ),
            stopped,
        )

    def shutdown(
        self,
        force=False,
    ):
        live_count = sum(
            1
            for p in self.procs
            if (
                p is not None
                and p.is_alive()
            )
        )

        for _ in range(
            live_count
        ):
            try:
                self.task_q.put_nowait(
                    None
                )
            except Exception:
                pass

        for p in self.procs:
            if p is None:
                continue

            try:
                p.join(
                    timeout=5
                )
            except Exception:
                pass

            if (
                force
                or p.is_alive()
            ):
                try:
                    p.terminate()
                except Exception:
                    pass


# Close a previous Cell-4 service if this cell is rerun.
if "_VOX_SERVICE" in globals():
    try:
        _VOX_SERVICE.shutdown(
            force=True
        )
    except Exception:
        pass

print(
    "Loading VoxCPM2 on T4 GPUs sequentially..."
)

_VOX_SERVICE = DualGPUVoxCPM(
    MODEL_DIR,
    gpu_ids=(0, 1),
    optimize=WORKER_OPTIMIZE,
    startup_timeout=(
        WORKER_STARTUP_TIMEOUT
    ),
)

print(
    "✓ Dual-T4 service ready"
)

# ------------------------------------------------------------
# DATASET SYNC
# ------------------------------------------------------------

def sync_library_to_dataset(
    handle,
):
    """
    Sync only permanent metadata + permanent reference WAVs.
    Generated audio, imports and cookies are excluded.
    """
    handle = (
        handle or ""
    ).strip()

    if "/" not in handle:
        raise ValueError(
            "Dataset handle must "
            "look like "
            "username/dataset-slug"
        )

    import kagglehub

    sync_dir = Path(
        "/kaggle/working/"
        "voxcpm_voice_dataset_sync"
    )

    shutil.rmtree(
        sync_dir,
        ignore_errors=True,
    )

    (
        sync_dir
        / "voices"
    ).mkdir(
        parents=True,
        exist_ok=True,
    )

    if META_FILE.exists():
        shutil.copy2(
            META_FILE,
            sync_dir
            / "library.json",
        )

    for vid in valid_voice_ids():
        src = voice_path(
            vid
        )

        shutil.copy2(
            src,
            sync_dir
            / "voices"
            / src.name,
        )

    kagglehub.dataset_upload(
        handle,
        str(sync_dir),
        version_notes=(
            "Voice Studio sync "
            + time.strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        ),
    )

# ------------------------------------------------------------
# UI STATE
# ------------------------------------------------------------

STATE = {
    "audition_path": None,
    "audition_text": None,
    "audition_direction": None,
    "audition_profile_id": None,

    "clone_path": None,
    "clone_transcript": "",
    "clone_source": "",

    "clone_busy": False,
    "audition_busy": False,
    "generation_busy": False,

    "cookie_file": None,
    "stop_event": None,
}

# ------------------------------------------------------------
# CSS / HEADER
# ------------------------------------------------------------

studio_css = widgets.HTML(
    r"""
<style>
.vox-card {
    background: #181818;
    color: #ffffff;
    border: 1px solid #343434;
    border-radius: 14px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
}
.vox-title {
    font-size: 27px;
    font-weight: 760;
    color: #ffffff;
}
.vox-sub {
    color: #aaaaaa;
    font-size: 13px;
    margin-top: 5px;
}
.vox-section {
    color: inherit;
    font-size: 18px;
    font-weight: 700;
    margin: 10px 0 8px 0;
}
.vox-note {
    color: #999999;
    font-size: 12px;
    line-height: 1.55;
}
.widget-button {
    border-radius: 10px !important;
    font-weight: 650 !important;
}
.widget-textarea textarea,
.widget-text input,
.widget-dropdown select {
    border-radius: 9px !important;
}
</style>
"""
)

header = widgets.HTML(
    r"""
<div class="vox-card">
  <div class="vox-title">
    🎙️ VoxCPM2 Voice Studio
  </div>
  <div class="vox-sub">
    Dual T4 • Burmese long-form • live chunk streaming • voice design • cloning
  </div>
</div>
"""
)

# ------------------------------------------------------------
# SHARED VOICE SELECTOR
# ------------------------------------------------------------

saved_voice = widgets.Dropdown(
    description="Voice",
    layout=widgets.Layout(
        width="560px"
    ),
)

play_selected_btn = widgets.Button(
    description="▶ Play",
    layout=widgets.Layout(
        width="95px"
    ),
)

delete_selected_btn = widgets.Button(
    description="🗑 Delete",
    button_style="danger",
    layout=widgets.Layout(
        width="110px"
    ),
)

library_output = widgets.Output()


def selected_voice_id():
    return (
        saved_voice.value
        or ""
    )


def refresh_saved_voices(
    select_id=None,
):
    ids = valid_voice_ids()

    ids.sort(
        key=lambda vid: (
            library[
                "voices"
            ][vid].get(
                "created",
                "",
            )
        ),
        reverse=True,
    )

    if ids:
        saved_voice.options = [
            (
                (
                    f"{library['voices'][vid].get('name', vid)}"
                    f"   •   "
                    f"{library['voices'][vid].get('created', '')}"
                ),
                vid,
            )
            for vid in ids
        ]

        desired = (
            select_id
            or library.get(
                "active_voice_id"
            )
            or ids[0]
        )

        saved_voice.value = (
            desired
            if desired in ids
            else ids[0]
        )

        play_selected_btn.disabled = False
        delete_selected_btn.disabled = False

    else:
        saved_voice.options = [
            (
                "— no saved voice yet —",
                "",
            )
        ]

        saved_voice.value = ""
        play_selected_btn.disabled = True
        delete_selected_btn.disabled = True


def on_voice_change(
    change=None,
):
    vid = selected_voice_id()

    if (
        vid
        and vid
        in library["voices"]
    ):
        library[
            "active_voice_id"
        ] = vid

        save_library()


saved_voice.observe(
    on_voice_change,
    names="value",
)


def on_play_selected(_):
    library_output.clear_output(
        wait=True
    )

    vid = selected_voice_id()

    if not vid:
        library_output.append_stdout(
            "No voice selected.\n"
        )
        return

    entry = (
        library["voices"][vid]
    )

    path = voice_path(
        vid
    )

    library_output.append_stdout(
        f"{entry.get('name', vid)}\n"
        f"Source: "
        f"{entry.get('source_type', 'saved voice')}\n\n"
    )

    library_output.append_display_data(
        Audio(
            filename=str(path),
            autoplay=False,
        )
    )


def on_delete_selected(_):
    library_output.clear_output(
        wait=True
    )

    if STATE[
        "generation_busy"
    ]:
        library_output.append_stdout(
            "Generation is running. "
            "Delete the voice after "
            "generation finishes.\n"
        )
        return

    vid = selected_voice_id()

    if not vid:
        return

    entry = (
        library["voices"]
        .get(
            vid,
            {},
        )
    )

    p = (
        LIBRARY_ROOT
        / entry.get(
            "file",
            "",
        )
    )

    try:
        if p.exists():
            p.unlink()
    except Exception:
        pass

    library["voices"].pop(
        vid,
        None,
    )

    if (
        library.get(
            "active_voice_id"
        )
        == vid
    ):
        library[
            "active_voice_id"
        ] = None

    save_library()
    refresh_saved_voices()

    library_output.append_stdout(
        "✓ Voice deleted "
        "from working library.\n"
    )


play_selected_btn.on_click(
    on_play_selected
)

delete_selected_btn.on_click(
    on_delete_selected
)

# ------------------------------------------------------------
# NOTEBOOK PLAYER / DOWNLOAD HELPERS
# ------------------------------------------------------------

def notebook_file_href(path):
    """
    Extract the notebook-served URL used by IPython FileLink.
    """
    path = str(path)

    try:
        rendered = FileLink(path)._repr_html_()

        match = re.search(
            r"href=['\"]([^'\"]+)['\"]",
            rendered,
        )

        if match:
            return match.group(1)

    except Exception:
        pass

    return path


def make_static_audio_card(
    path,
    title="Audio",
    mime="audio/wav",
):
    """
    Short-file player for profile tests / clone references.
    Includes seconds, three-dot menu, playback speed and download.
    """
    path = Path(path)

    player_id = (
        "vox_static_"
        + uuid.uuid4().hex[:10]
    )

    menu_id = player_id + "_menu"
    audio_id = player_id + "_audio"

    audio_src = file_as_data_url(
        path,
        mime=mime,
    )

    download_href = notebook_file_href(
        path
    )

    html = f"""
<div class="vox-card" style="padding:14px; position:relative;">
  <div style="
      display:flex;
      align-items:center;
      justify-content:space-between;
      gap:10px;
      margin-bottom:8px;
  ">
    <div style="font-weight:700;">{title}</div>

    <div style="position:relative;">
      <button
        onclick="
          const m=document.getElementById('{menu_id}');
          m.style.display=(m.style.display==='block'?'none':'block');
        "
        style="
          border:0;
          background:transparent;
          color:inherit;
          font-size:22px;
          cursor:pointer;
          line-height:1;
        "
        title="More"
      >⋯</button>

      <div
        id="{menu_id}"
        style="
          display:none;
          position:absolute;
          right:0;
          top:28px;
          z-index:9999;
          min-width:180px;
          padding:10px;
          border-radius:10px;
          background:#222;
          color:#fff;
          box-shadow:0 6px 24px rgba(0,0,0,.35);
        "
      >
        <div style="font-size:12px; margin-bottom:5px;">
          Playback speed
        </div>

        <select
          onchange="
            document.getElementById('{audio_id}').playbackRate =
              parseFloat(this.value);
          "
          style="width:100%; margin-bottom:10px;"
        >
          <option value="0.75">0.75×</option>
          <option value="1" selected>1.00×</option>
          <option value="1.25">1.25×</option>
          <option value="1.5">1.50×</option>
          <option value="2">2.00×</option>
        </select>

        <a
          href="{download_href}"
          target="_blank"
          download
          style="
            display:block;
            color:#fff;
            text-decoration:none;
            padding:7px 0;
          "
        >
          ⬇ Download
        </a>
      </div>
    </div>
  </div>

  <audio
    id="{audio_id}"
    src="{audio_src}"
    controls
    preload="metadata"
    style="width:100%;"
  ></audio>
</div>
"""

    return HTML(html)


# ------------------------------------------------------------
# GENERATE TAB — CONTROLS
# ------------------------------------------------------------

gen_script = widgets.Textarea(
    value="",
    placeholder=(
        "Paste your Burmese "
        "or English script here..."
    ),
    layout=widgets.Layout(
        width="100%",
        height="300px",
    ),
)

delivery_preset = widgets.Dropdown(
    options=list(
        DELIVERY_PRESETS.keys()
    ),
    value=(
        "Natural / "
        "keep cloned delivery"
    ),
    description="Delivery",
    layout=widgets.Layout(
        width="560px"
    ),
)

custom_style = widgets.Text(
    value="",
    description="Custom",
    placeholder=(
        "tone/emotion only — "
        "e.g. calm, tense, cheerful"
    ),
    layout=widgets.Layout(
        width="100%"
    ),
)

speed_slider = widgets.FloatSlider(
    value=1.00,
    min=0.75,
    max=1.50,
    step=0.05,
    description="Speed",
    readout_format=".2f",
    continuous_update=False,
    layout=widgets.Layout(
        width="560px"
    ),
)

chunk_chars = widgets.IntSlider(
    value=220,
    min=120,
    max=360,
    step=10,
    description="Chunk chars",
    continuous_update=False,
    layout=widgets.Layout(
        width="560px"
    ),
)

join_pause = widgets.IntSlider(
    value=120,
    min=0,
    max=1000,
    step=20,
    description="Join pause",
    continuous_update=False,
    layout=widgets.Layout(
        width="560px"
    ),
)

trim_edges_checkbox = widgets.Checkbox(
    value=True,
    description=(
        "Trim generated edge silence "
        "before joining"
    ),
    indent=False,
)

trim_db = widgets.FloatSlider(
    value=-42.0,
    min=-60.0,
    max=-25.0,
    step=1.0,
    description="Silence dB",
    continuous_update=False,
    layout=widgets.Layout(
        width="560px"
    ),
)

cfg_slider = widgets.FloatSlider(
    value=2.0,
    min=1.0,
    max=3.0,
    step=0.1,
    description="CFG",
    continuous_update=False,
    layout=widgets.Layout(
        width="560px"
    ),
)

steps_slider = widgets.IntSlider(
    value=10,
    min=4,
    max=30,
    step=1,
    description="Steps",
    continuous_update=False,
    layout=widgets.Layout(
        width="560px"
    ),
)

generate_btn = widgets.Button(
    description="⚡ Generate + Stream",
    button_style="success",
    layout=widgets.Layout(
        width="285px",
        height="50px",
    ),
)

gen_status = widgets.HTML()

gen_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=1,
    description="Progress",
    bar_style="info",
    orientation="horizontal",
    layout=widgets.Layout(
        width="100%"
    ),
)

gen_text_note = widgets.HTML(
    "<div class='vox-note'>"
    "<b>Burmese-safe mode:</b> "
    "VoxCPM WeText normalization is OFF. "
    "Empty/punctuation-only chunks are filtered automatically."
    "</div>"
)

gen_stream_note = widgets.HTML(
    "<div class='vox-note'>"
    "<b>One live player only:</b> completed chunks are queued behind the "
    "audio you are already hearing. New chunks do <b>not</b> replace the "
    "current audio source, so they do not restart playback. If you catch "
    "up with generation, the player shows <b>Buffering — still generating</b>."
    "</div>"
)

generate_btn = widgets.Button(
    description="⚡ Generate + Stream",
    button_style="success",
    layout=widgets.Layout(
        width="285px",
        height="50px",
    ),
)

stop_btn = widgets.Button(
    description="■ Stop generating",
    button_style="danger",
    disabled=True,
    layout=widgets.Layout(
        width="190px",
        height="50px",
    ),
)

stream_player_output = widgets.Output()

# JavaScript commands are isolated from the visible player DOM.
stream_command_output = widgets.Output(
    layout=widgets.Layout(
        height="1px",
        min_height="1px",
        max_height="1px",
        overflow="hidden",
        opacity="0.0",
    )
)


def _send_stream_command(
    js,
):
    stream_command_output.append_display_data(
        js
    )


STREAM_PLAYER_ID = (
    "voxcpm_webaudio_"
    + uuid.uuid4().hex[:10]
)

STREAM_STATUS_ID = STREAM_PLAYER_ID + "_status"
STREAM_TIME_ID = STREAM_PLAYER_ID + "_time"
STREAM_PLAY_ID = STREAM_PLAYER_ID + "_play"
STREAM_BAR_ID = STREAM_PLAYER_ID + "_bar"
STREAM_MENU_ID = STREAM_PLAYER_ID + "_menu"
STREAM_SPEED_ID = STREAM_PLAYER_ID + "_speed"
STREAM_DOWNLOAD_ID = STREAM_PLAYER_ID + "_download"


def _stream_player_shell():
    return HTML(
        f"""
<div class="vox-card" style="padding:14px; position:relative;">
  <div style="
      display:flex;
      justify-content:space-between;
      align-items:center;
      gap:12px;
      margin-bottom:10px;
  ">
    <div style="font-weight:700;">
      🔴 Live / Final Voiceover
    </div>

    <div style="position:relative;">
      <button
        onclick="
          const m=document.getElementById('{STREAM_MENU_ID}');
          m.style.display=(m.style.display==='block'?'none':'block');
        "
        style="
          border:0;
          background:transparent;
          color:inherit;
          font-size:24px;
          cursor:pointer;
          line-height:1;
        "
        title="More"
      >⋯</button>

      <div
        id="{STREAM_MENU_ID}"
        style="
          display:none;
          position:absolute;
          right:0;
          top:28px;
          z-index:9999;
          width:205px;
          padding:11px;
          border-radius:10px;
          background:#222;
          color:#fff;
          box-shadow:0 6px 24px rgba(0,0,0,.35);
        "
      >
        <div style="font-size:12px;margin-bottom:5px;">
          Playback speed
        </div>

        <select
          id="{STREAM_SPEED_ID}"
          style="width:100%;margin-bottom:10px;"
        >
          <option value="0.75">0.75×</option>
          <option value="1" selected>1.00×</option>
          <option value="1.25">1.25×</option>
          <option value="1.5">1.50×</option>
          <option value="2">2.00×</option>
        </select>

        <a
          id="{STREAM_DOWNLOAD_ID}"
          href="#"
          style="
            display:block;
            color:#777;
            pointer-events:none;
            text-decoration:none;
            padding:7px 0;
          "
        >
          ⬇ Download final WAV
        </a>
      </div>
    </div>
  </div>

  <div style="
      display:flex;
      align-items:center;
      gap:12px;
      width:100%;
  ">
    <button
      id="{STREAM_PLAY_ID}"
      style="
        width:46px;
        height:38px;
        border-radius:20px;
        border:0;
        cursor:pointer;
        font-size:17px;
      "
    >▶</button>

    <input
      id="{STREAM_BAR_ID}"
      type="range"
      min="0"
      max="1"
      value="0"
      step="0.01"
      disabled
      style="flex:1;"
    />
  </div>

  <div
    id="{STREAM_TIME_ID}"
    style="
      margin-top:8px;
      font-size:12px;
      font-variant-numeric:tabular-nums;
      color:#bdbdbd;
    "
  >
    00:00 / 00:00 generated
  </div>

  <div
    id="{STREAM_STATUS_ID}"
    style="
      margin-top:5px;
      color:#bdbdbd;
      font-size:12px;
    "
  >
    Waiting for the first generated audio…
  </div>
</div>
"""
    )


def _init_stream_player(
    total_chunks,
):
    # Only this function creates/recreates the visible player.
    stream_player_output.clear_output(
        wait=True
    )

    stream_command_output.clear_output(
        wait=True
    )

    stream_player_output.append_display_data(
        _stream_player_shell()
    )

    js = Javascript(
        f"""
(() => {{
  const id = {json.dumps(STREAM_PLAYER_ID)};
  const playId = {json.dumps(STREAM_PLAY_ID)};
  const statusId = {json.dumps(STREAM_STATUS_ID)};
  const timeId = {json.dumps(STREAM_TIME_ID)};
  const barId = {json.dumps(STREAM_BAR_ID)};
  const speedId = {json.dumps(STREAM_SPEED_ID)};
  const total = {int(total_chunks)};

  window.__voxWebAudio =
    window.__voxWebAudio || {{}};

  const playBtn = document.getElementById(playId);
  const statusEl = document.getElementById(statusId);
  const timeEl = document.getElementById(timeId);
  const barEl = document.getElementById(barId);
  const speedEl = document.getElementById(speedId);

  const AudioCtx =
    window.AudioContext || window.webkitAudioContext;

  if (!AudioCtx || !playBtn) {{
    if (statusEl) {{
      statusEl.textContent =
        "Web Audio is unavailable in this browser.";
    }}
    return;
  }}

  const state = {{
    ctx: new AudioCtx(),
    buffers: {{}},
    durations: {{}},
    current: 0,
    source: null,
    playing: false,
    wantPlay: false,
    buffering: false,
    manualStop: false,
    offset: 0,
    startedAt: 0,
    startedOffset: 0,
    rate: 1.0,
    total: total,
    generatedCount: 0,
    generationDone: false,
    finalCount: null,
    stopped: false
  }};

  window.__voxWebAudio[id] = state;

  const fmt = (sec) => {{
    sec = Math.max(0, Math.floor(Number(sec) || 0));
    const m = Math.floor(sec / 60);
    const s = sec % 60;

    return (
      String(m).padStart(2, "0")
      + ":"
      + String(s).padStart(2, "0")
    );
  }};

  const contiguousCount = () => {{
    let i = 0;

    while (state.buffers[i]) {{
      i += 1;
    }}

    return i;
  }};

  const secondsBefore = (idx) => {{
    let value = 0;

    for (let i = 0; i < idx; i++) {{
      value += Number(
        state.durations[i] || 0
      );
    }}

    return value;
  }};

  const availableSeconds = () => {{
    return secondsBefore(
      contiguousCount()
    );
  }};

  const currentOffset = () => {{
    if (!state.playing || !state.source) {{
      return state.offset;
    }}

    const elapsed =
      (state.ctx.currentTime - state.startedAt)
      * state.rate;

    const duration =
      Number(
        state.durations[state.current] || 0
      );

    return Math.min(
      duration,
      state.startedOffset + elapsed
    );
  }};

  const playedSeconds = () => {{
    return (
      secondsBefore(state.current)
      + currentOffset()
    );
  }};

  const setStatus = (message) => {{
    if (statusEl) {{
      statusEl.textContent = message;
    }}
  }};

  const updateUi = () => {{
    const played = playedSeconds();
    const available = availableSeconds();

    if (timeEl) {{
      timeEl.textContent =
        `${{fmt(played)}} / ${{fmt(available)}} generated`;
    }}

    if (barEl) {{
      barEl.max = Math.max(
        1,
        available
      );

      barEl.value = Math.min(
        played,
        available
      );
    }}

    if (playBtn) {{
      playBtn.textContent =
        state.playing
          ? "⏸"
          : "▶";
    }}
  }};

  const stopSource = () => {{
    if (!state.source) return;

    state.manualStop = true;

    try {{
      state.source.onended = null;
      state.source.stop();
    }} catch (_) {{}}

    try {{
      state.source.disconnect();
    }} catch (_) {{}}

    state.source = null;
    state.manualStop = false;
  }};

  const pauseCurrent = () => {{
    if (!state.playing) return;

    state.offset = currentOffset();
    state.playing = false;
    state.wantPlay = false;

    stopSource();

    setStatus(
      `Paused • ${{state.generatedCount}}/${{state.total}} chunks generated`
    );

    updateUi();
  }};

  const playCurrent = async () => {{
    await state.ctx.resume();

    const limit =
      (
        state.generationDone
        && state.finalCount !== null
      )
        ? state.finalCount
        : state.total;

    if (state.current >= limit) {{
      state.playing = false;
      state.wantPlay = false;

      setStatus(
        state.stopped
          ? "■ Stopped • partial audio complete"
          : "✓ Playback complete"
      );

      updateUi();
      return;
    }}

    const buffer =
      state.buffers[
        state.current
      ];

    if (!buffer) {{
      state.playing = false;
      state.wantPlay = true;
      state.buffering = true;

      setStatus(
        `⏳ Buffering — still generating… ` +
        `(${{state.generatedCount}}/${{state.total}} chunks ready)`
      );

      updateUi();
      return;
    }}

    state.buffering = false;

    const duration =
      Number(
        state.durations[state.current]
        || buffer.duration
      );

    if (state.offset >= duration - 0.01) {{
      state.current += 1;
      state.offset = 0;

      await playCurrent();
      return;
    }}

    const source =
      state.ctx.createBufferSource();

    source.buffer = buffer;
    source.playbackRate.value =
      state.rate;

    source.connect(
      state.ctx.destination
    );

    state.source = source;
    state.startedAt =
      state.ctx.currentTime;
    state.startedOffset =
      state.offset;
    state.playing = true;
    state.wantPlay = true;

    source.onended = async () => {{
      if (state.source !== source) {{
        return;
      }}

      state.source = null;
      state.playing = false;
      state.offset = 0;
      state.current += 1;

      updateUi();

      if (state.wantPlay) {{
        await playCurrent();
      }}
    }};

    source.start(
      0,
      state.offset
    );

    setStatus(
      `▶ Playing • ${{state.generatedCount}}/${{state.total}} chunks generated`
    );

    updateUi();
  }};

  const togglePlayback = async () => {{
    if (state.playing) {{
      pauseCurrent();
      return;
    }}

    state.wantPlay = true;
    await playCurrent();
  }};

  const changeRate = async () => {{
    const newRate =
      parseFloat(
        speedEl ? speedEl.value : "1"
      ) || 1;

    const wasPlaying =
      state.playing;

    if (wasPlaying) {{
      state.offset =
        currentOffset();

      state.playing = false;
      stopSource();
    }}

    state.rate =
      newRate;

    if (wasPlaying) {{
      state.wantPlay = true;
      await playCurrent();
    }}

    updateUi();
  }};

  state.addSegment = async (
    idx,
    dataUrl,
    declaredDuration
  ) => {{
    if (state.buffers[idx]) {{
      return;
    }}

    // Decode the newly generated chunk in the background.
    // This never stops or replaces the current AudioBufferSource.
    const response =
      await fetch(dataUrl);

    const arrayBuffer =
      await response.arrayBuffer();

    const decoded =
      await state.ctx.decodeAudioData(
        arrayBuffer.slice(0)
      );

    state.buffers[idx] =
      decoded;

    state.durations[idx] =
      Number(
        declaredDuration
        || decoded.duration
      );

    state.generatedCount =
      contiguousCount();

    updateUi();

    if (
      state.buffering
      && state.wantPlay
      && state.buffers[state.current]
    ) {{
      await playCurrent();
    }} else if (!state.playing) {{
      setStatus(
        `${{state.generatedCount}}/${{state.total}} chunks ready • press Play`
      );
    }}
  }};

  state.updateUi = updateUi;

  playBtn.addEventListener(
    "click",
    togglePlayback
  );

  if (speedEl) {{
    speedEl.addEventListener(
      "change",
      changeRate
    );
  }}

  setInterval(
    updateUi,
    250
  );

  updateUi();
}})();
"""
    )

    _send_stream_command(
        js
    )


def _queue_stream_segment(
    index,
    total_chunks,
    data_url,
    duration_seconds,
    contiguous_count,
):
    # Add/decode only. It does NOT replace the sound currently playing.
    js = Javascript(
        f"""
(async () => {{
  const id = {json.dumps(STREAM_PLAYER_ID)};
  const state =
    (window.__voxWebAudio || {{}})[id];

  if (!state || !state.addSegment) {{
    return;
  }}

  await state.addSegment(
    {int(index)},
    {json.dumps(data_url)},
    {float(duration_seconds)}
  );
}})();
"""
    )

    _send_stream_command(
        js
    )


def _finish_stream_player(
    download_href,
    download_name,
    final_count,
    total_chunks,
    stopped=False,
):
    status_text = (
        "■ Generation stopped • partial WAV ready"
        if stopped
        else "✓ 100% generated • final WAV ready"
    )

    js = Javascript(
        f"""
(() => {{
  const id = {json.dumps(STREAM_PLAYER_ID)};
  const did = {json.dumps(STREAM_DOWNLOAD_ID)};
  const sid = {json.dumps(STREAM_STATUS_ID)};

  const state =
    (window.__voxWebAudio || {{}})[id];

  const download =
    document.getElementById(did);

  const status =
    document.getElementById(sid);

  if (state) {{
    state.generationDone = true;
    state.finalCount = {int(final_count)};
    state.generatedCount = {int(final_count)};
    state.stopped = {str(bool(stopped)).lower()};

    if (state.updateUi) {{
      state.updateUi();
    }}
  }}

  if (download) {{
    download.href =
      {json.dumps(download_href)};
    download.download =
      {json.dumps(download_name)};
    download.style.color = "#fff";
    download.style.pointerEvents = "auto";
  }}

  if (status) {{
    status.textContent =
      {json.dumps(status_text)};
  }}
}})();
"""
    )

    _send_stream_command(
        js
    )


def _fail_stream_player(
    message,
    final_count=0,
):
    js = Javascript(
        f"""
(() => {{
  const id = {json.dumps(STREAM_PLAYER_ID)};
  const sid = {json.dumps(STREAM_STATUS_ID)};

  const state =
    (window.__voxWebAudio || {{}})[id];

  const status =
    document.getElementById(sid);

  if (state) {{
    state.generationDone = true;
    state.finalCount = {int(final_count)};

    if (state.updateUi) {{
      state.updateUi();
    }}
  }}

  if (status) {{
    status.textContent =
      {json.dumps("Generation stopped with error: " + str(message))};
  }}
}})();
"""
    )

    _send_stream_command(
        js
    )


def build_delivery_control():
    base = (
        DELIVERY_PRESETS
        .get(
            delivery_preset.value,
            "",
        )
        .strip()
    )

    custom = (
        custom_style.value
        .strip()
    )

    return ", ".join(
        [
            x
            for x
            in (
                base,
                custom,
            )
            if x
        ]
    )


def make_generation_kwargs(
    chunk,
    ref_path,
    reference_text,
    control,
):
    common = {
        "cfg_value": float(
            cfg_slider.value
        ),
        "inference_timesteps": int(
            steps_slider.value
        ),
        "retry_badcase": True,
        "normalize": False,
    }

    if control:
        common.update(
            {
                "text": (
                    f"({control})"
                    f"{chunk}"
                ),
                "reference_wav_path": str(
                    ref_path
                ),
            }
        )

    else:
        common.update(
            {
                "text": chunk,
                "reference_wav_path": str(
                    ref_path
                ),
            }
        )

        if reference_text:
            common.update(
                {
                    "prompt_wav_path": str(
                        ref_path
                    ),
                    "prompt_text": (
                        reference_text
                    ),
                }
            )

    return common


def _build_downloadable_wav(
    prepared,
    count,
    stamp,
    stopped=False,
    errored=False,
):
    if count <= 0:
        return None, 0, 0

    ordered = [
        prepared[i]
        for i
        in range(count)
    ]

    if errored:
        name = (
            f"voiceover_partial_error_"
            f"{stamp}.wav"
        )
    elif stopped:
        name = (
            f"voiceover_stopped_"
            f"{stamp}.wav"
        )
    else:
        name = (
            f"voiceover_"
            f"{stamp}.wav"
        )

    final_path = (
        GENERATED_DIR
        / name
    )

    _, sr, samples = (
        concatenate_prepared_chunks(
            prepared_paths=ordered,
            output_path=final_path,
            pause_ms=int(
                join_pause.value
            ),
        )
    )

    return (
        final_path,
        sr,
        samples,
    )


def _generate_long_job(
    stop_event,
):
    prepared = {}
    next_stream_index = 0

    stamp = time.strftime(
        "%Y%m%d_%H%M%S"
    )

    total_chunks = 0

    try:
        vid = selected_voice_id()

        if not vid:
            raise ValueError(
                "Save/select a permanent "
                "voice first."
            )

        script = sanitize_tts_text(
            gen_script.value
        )

        if not script:
            raise ValueError(
                "Paste a script first."
            )

        entry = (
            library["voices"][vid]
        )

        ref = voice_path(
            vid
        )

        if not ref.exists():
            raise FileNotFoundError(
                "Selected reference "
                "WAV is missing."
            )

        reference_text = (
            entry
            .get(
                "transcript",
                "",
            )
            or ""
        ).strip()

        chunks = split_long_text(
            script,
            max_chars=int(
                chunk_chars.value
            ),
        )

        chunks = clean_chunks_for_tts(
            chunks
        )

        if not chunks:
            raise ValueError(
                "No speakable text chunks "
                "were created."
            )

        total_chunks = len(
            chunks
        )

        control = (
            build_delivery_control()
        )

        batch_dir = (
            GENERATED_DIR
            / (
                f"batch_{stamp}_"
                f"{uuid.uuid4().hex[:6]}"
            )
        )

        batch_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        stream_dir = (
            batch_dir
            / "stream_ready"
        )

        stream_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        gen_progress.min = 0
        gen_progress.max = total_chunks
        gen_progress.value = 0
        gen_progress.bar_style = "info"

        _init_stream_player(
            total_chunks
        )

        gen_status.value = (
            f"<b>Generating…</b> "
            f"0/{total_chunks} chunks • "
            f"two T4 workers • "
            f"{entry.get('name', vid)}"
        )

        tasks = []

        for i, chunk in enumerate(
            chunks
        ):
            out = (
                batch_dir
                / f"chunk_{i:04d}.wav"
            )

            tasks.append(
                {
                    "chunk_index": i,
                    "output_path": str(
                        out
                    ),
                    "kwargs": (
                        make_generation_kwargs(
                            chunk=chunk,
                            ref_path=ref,
                            reference_text=reference_text,
                            control=control,
                        )
                    ),
                }
            )

        arrived = {}
        callback_lock = (
            threading.Lock()
        )

        def on_chunk_finished(
            msg,
            finished_count,
            total_count,
        ):
            nonlocal next_stream_index

            idx = int(
                msg["chunk_index"]
            )

            with callback_lock:
                arrived[idx] = msg

                gen_progress.value = int(
                    finished_count
                )

                while (
                    next_stream_index
                    in arrived
                ):
                    ready = (
                        arrived[
                            next_stream_index
                        ]
                    )

                    part_path = (
                        stream_dir
                        / (
                            f"part_"
                            f"{next_stream_index:04d}"
                            f".wav"
                        )
                    )

                    _, sr, sample_count = (
                        prepare_stream_chunk(
                            raw_path=(
                                ready[
                                    "output_path"
                                ]
                            ),
                            output_path=(
                                part_path
                            ),
                            speed=float(
                                speed_slider.value
                            ),
                            trim_edges=bool(
                                trim_edges_checkbox.value
                            ),
                            trim_db=float(
                                trim_db.value
                            ),
                        )
                    )

                    prepared[
                        next_stream_index
                    ] = str(
                        part_path
                    )

                    stream_mp3 = (
                        stream_dir
                        / (
                            f"stream_"
                            f"{next_stream_index:04d}"
                            f".mp3"
                        )
                    )

                    make_stream_segment_mp3(
                        prepared_wav=(
                            part_path
                        ),
                        output_mp3=(
                            stream_mp3
                        ),
                        pause_ms=int(
                            join_pause.value
                        ),
                        add_pause=True,
                    )

                    audio_seconds = (
                        float(sample_count)
                        / float(sr)
                    )

                    stream_duration = (
                        audio_seconds
                        + (
                            float(
                                join_pause.value
                            )
                            / 1000.0
                        )
                    )

                    _queue_stream_segment(
                        index=(
                            next_stream_index
                        ),
                        total_chunks=(
                            total_count
                        ),
                        data_url=(
                            file_as_data_url(
                                stream_mp3,
                                mime="audio/mpeg",
                            )
                        ),
                        duration_seconds=(
                            stream_duration
                        ),
                        contiguous_count=(
                            next_stream_index
                            + 1
                        ),
                    )

                    next_stream_index += 1

                gen_status.value = (
                    f"<b>Generating…</b> "
                    f"{finished_count}/"
                    f"{total_count} worker "
                    f"tasks finished • "
                    f"{next_stream_index}/"
                    f"{total_count} streamable"
                )

        start = time.time()

        results, stopped = (
            _VOX_SERVICE.run(
                tasks,
                on_result=(
                    on_chunk_finished
                ),
                stop_event=(
                    stop_event
                ),
            )
        )

        completed_count = (
            next_stream_index
        )

        if completed_count <= 0:
            if stopped:
                gen_status.value = (
                    "<b>■ Stopped</b> "
                    "before any audio finished."
                )

                _fail_stream_player(
                    "Stopped before first chunk",
                    final_count=0,
                )

                return

            raise RuntimeError(
                "Generation ended without "
                "a completed audio chunk."
            )

        final_path, sr, samples = (
            _build_downloadable_wav(
                prepared=prepared,
                count=completed_count,
                stamp=stamp,
                stopped=stopped,
            )
        )

        elapsed = (
            time.time()
            - start
        )

        duration = (
            samples / sr
            if sr
            else 0
        )

        _finish_stream_player(
            download_href=(
                notebook_file_href(
                    final_path
                )
            ),
            download_name=(
                final_path.name
            ),
            final_count=(
                completed_count
            ),
            total_chunks=(
                total_chunks
            ),
            stopped=(
                stopped
            ),
        )

        if stopped:
            gen_progress.value = (
                completed_count
            )

            gen_progress.bar_style = (
                "warning"
            )

            gen_status.value = (
                f"<b>■ Stopped</b> • "
                f"{completed_count}/"
                f"{total_chunks} chunks saved • "
                f"{duration:.1f}s partial audio"
            )

        else:
            gen_progress.value = (
                total_chunks
            )

            gen_progress.bar_style = (
                "success"
            )

            used = sorted(
                {
                    r["gpu"]
                    for r
                    in results
                }
            )

            gen_status.value = (
                f"<b>✓ Finished</b> • "
                f"{elapsed:.1f}s processing • "
                f"{duration:.1f}s audio • "
                f"GPUs {used}"
            )

    except Exception as e:
        completed_count = (
            next_stream_index
        )

        if completed_count > 0:
            try:
                partial_path, _, _ = (
                    _build_downloadable_wav(
                        prepared=prepared,
                        count=completed_count,
                        stamp=stamp,
                        errored=True,
                    )
                )

                _finish_stream_player(
                    download_href=(
                        notebook_file_href(
                            partial_path
                        )
                    ),
                    download_name=(
                        partial_path.name
                    ),
                    final_count=(
                        completed_count
                    ),
                    total_chunks=(
                        total_chunks
                        or completed_count
                    ),
                    stopped=True,
                )

            except Exception:
                _fail_stream_player(
                    str(e),
                    final_count=(
                        completed_count
                    ),
                )

        else:
            _fail_stream_player(
                str(e),
                final_count=0,
            )

        gen_progress.bar_style = (
            "danger"
        )

        gen_status.value = (
            "<span style='color:#ff5f56'>"
            "<b>Generation failed:</b> "
            f"{str(e)}"
            "</span>"
        )

    finally:
        STATE[
            "generation_busy"
        ] = False

        STATE[
            "stop_event"
        ] = None

        generate_btn.disabled = False

        stop_btn.disabled = True
        stop_btn.description = (
            "■ Stop generating"
        )


def on_generate(_):
    if STATE[
        "generation_busy"
    ]:
        return

    if STATE.get(
        "clone_busy"
    ):
        gen_status.value = (
            "<span style='color:#b7791f'>"
            "Voice Clone is using GPU 1 for Whisper. "
            "Wait until transcription finishes and GPU 1 reloads VoxCPM2."
            "</span>"
        )
        return

    stop_event = (
        threading.Event()
    )

    STATE[
        "stop_event"
    ] = stop_event

    STATE[
        "generation_busy"
    ] = True

    generate_btn.disabled = True
    stop_btn.disabled = False
    stop_btn.description = (
        "■ Stop generating"
    )

    threading.Thread(
        target=_generate_long_job,
        args=(
            stop_event,
        ),
        daemon=True,
    ).start()


def on_stop_generation(_):
    event = STATE.get(
        "stop_event"
    )

    if (
        not STATE.get(
            "generation_busy"
        )
        or event is None
    ):
        return

    event.set()

    stop_btn.disabled = True
    stop_btn.description = (
        "Stopping…"
    )

    gen_status.value = (
        "<b>Stopping…</b> "
        "No new chunks will start. "
        "The one or two chunks already on the GPUs "
        "will finish, then the partial WAV will be saved."
    )


generate_btn.on_click(
    on_generate
)

stop_btn.on_click(
    on_stop_generation
)

generate_advanced = widgets.Accordion(
    children=[
        widgets.VBox(
            [
                chunk_chars,
                join_pause,
                trim_edges_checkbox,
                trim_db,
                cfg_slider,
                steps_slider,
            ]
        )
    ],
    selected_index=None,
)

generate_advanced.set_title(
    0,
    "Advanced generation / chunk settings",
)

generate_tab = widgets.VBox(
    [
        widgets.HTML(
            '<div class="vox-section">'
            "Voice"
            "</div>"
        ),
        widgets.HBox(
            [
                saved_voice,
                play_selected_btn,
            ]
        ),
        widgets.HTML(
            '<div class="vox-section">'
            "Script"
            "</div>"
        ),
        gen_script,
        widgets.HTML(
            '<div class="vox-section">'
            "Voice settings"
            "</div>"
        ),
        delivery_preset,
        custom_style,
        speed_slider,
        widgets.HTML(
            '<div class="vox-note">'
            "<b>Generation Speed</b> changes the saved audio itself. "
            "The ⋯ menu on the player changes playback speed only."
            "</div>"
        ),
        generate_advanced,
        gen_text_note,
        widgets.HBox(
            [
                generate_btn,
                stop_btn,
            ],
            layout=widgets.Layout(
                gap="10px",
                flex_flow="row wrap",
            ),
        ),
        gen_status,
        gen_progress,
        gen_stream_note,
        stream_player_output,
        stream_command_output,
    ]
)

# ------------------------------------------------------------
# VOICE DESIGN TAB
# ------------------------------------------------------------

profile_select = widgets.Dropdown(
    options=[
        (
            p["name"],
            pid,
        )
        for pid, p
        in VOICE_PROFILES.items()
    ],
    value="u_aung",
    description="Profile",
    layout=widgets.Layout(
        width="640px"
    ),
)

profile_info = widgets.HTML()

design_audition_text = widgets.Textarea(
    value=(
        "မင်္ဂလာပါ။ "
        "ဒီအသံက သဘာဝကျပြီး "
        "နားထောင်ရလွယ်ကူတဲ့ "
        "မြန်မာစကားပြောသံ "
        "ဖြစ်စေချင်ပါတယ်။"
    ),
    placeholder=(
        "Use one to three "
        "sentences..."
    ),
    layout=widgets.Layout(
        width="100%",
        height="120px",
    ),
)

design_name = widgets.Text(
    value="My Burmese Voice",
    description="Save name",
    layout=widgets.Layout(
        width="640px"
    ),
)

design_btn = widgets.Button(
    description=(
        "🎲 Generate profile candidate"
    ),
    button_style="success",
    layout=widgets.Layout(
        width="280px",
        height="46px",
    ),
)

save_design_btn = widgets.Button(
    description="🔒 Save candidate",
    button_style="warning",
    disabled=True,
    layout=widgets.Layout(
        width="220px",
        height="46px",
    ),
)

design_status = widgets.HTML()
design_output = widgets.Output()

save_design_btn.description = (
    "💾 Save this voice"
)

save_design_btn.layout = widgets.Layout(
    width="220px",
    height="44px",
)

save_design_btn.disabled = True

design_player_box = widgets.VBox(
    [
        design_output,
        save_design_btn,
    ]
)


def refresh_profile(
    change=None,
):
    profile = (
        VOICE_PROFILES[
            profile_select.value
        ]
    )

    profile_info.value = (
        "<div class='vox-card'>"
        f"<b>{profile['name']}</b>"
        "<br>"
        "<span class='vox-note'>"
        f"{profile['design']}"
        "</span>"
        "</div>"
    )


profile_select.observe(
    refresh_profile,
    names="value",
)

refresh_profile()


def _audition_job():
    try:
        pid = (
            profile_select.value
        )

        profile = (
            VOICE_PROFILES[pid]
        )

        text = sanitize_tts_text(
            design_audition_text.value
        )

        if not has_speakable_content(
            text
        ):
            raise ValueError(
                "Enter a short "
                "audition script."
            )

        direction = profile[
            "design"
        ]

        out = (
            GENERATED_DIR
            / (
                f"audition_"
                f"{pid}_"
                f"{int(time.time())}_"
                f"{uuid.uuid4().hex[:5]}"
                f".wav"
            )
        )

        design_status.value = (
            f"<b>Generating:</b> "
            f"{profile['name']}"
        )

        results, _ = (
            _VOX_SERVICE.run(
                [
                    {
                        "chunk_index": 0,
                        "output_path": str(
                            out
                        ),
                        "kwargs": {
                            "text": (
                                f"({direction})"
                                f"{text}"
                            ),
                            "cfg_value": 2.0,
                            "inference_timesteps": 10,
                            "retry_badcase": True,
                            "normalize": False,
                        },
                    }
                ]
            )
        )

        result = results[0]

        STATE[
            "audition_path"
        ] = str(out)

        STATE[
            "audition_text"
        ] = text

        STATE[
            "audition_direction"
        ] = direction

        STATE[
            "audition_profile_id"
        ] = pid

        design_status.value = (
            f"<b>✓ Candidate ready</b> "
            f"• GPU "
            f"{result.get('gpu')}"
        )

        design_output.clear_output(
            wait=True
        )

        design_output.append_display_data(
            make_static_audio_card(
                path=out,
                title=(
                    "🎧 Test voice candidate"
                ),
                mime="audio/wav",
            )
        )

        save_design_btn.disabled = False

    except Exception as e:
        design_status.value = (
            "<span style='color:#ff5f56'>"
            "<b>Failed:</b> "
            f"{str(e)}"
            "</span>"
        )

    finally:
        STATE[
            "audition_busy"
        ] = False

        design_btn.disabled = False


def on_design(_):
    if STATE[
        "audition_busy"
    ]:
        return

    if STATE.get(
        "clone_busy"
    ):
        design_status.value = (
            "<span style='color:#b7791f'>"
            "Voice Clone is using GPU 1 for Whisper. "
            "Wait for transcription and VoxCPM2 reload to finish."
            "</span>"
        )
        return

    if STATE.get(
        "generation_busy"
    ):
        design_status.value = (
            "<span style='color:#b7791f'>"
            "Wait for long-form generation to finish or stop it first."
            "</span>"
        )
        return

    STATE[
        "audition_busy"
    ] = True

    design_btn.disabled = True
    save_design_btn.disabled = True
    save_design_btn.description = (
        "💾 Save this voice"
    )

    design_output.clear_output(
        wait=True
    )

    threading.Thread(
        target=_audition_job,
        daemon=True,
    ).start()


def on_save_design(_):
    candidate = STATE.get(
        "audition_path"
    )

    if (
        not candidate
        or not Path(
            candidate
        ).exists()
    ):
        design_status.value = (
            "<span style='color:#ff5f56'>"
            "Generate a candidate first."
            "</span>"
        )
        return

    name = (
        design_name.value.strip()
        or "Designed Voice"
    )

    vid = (
        uuid.uuid4().hex[:12]
    )

    dest = (
        VOICE_DIR
        / f"{vid}.wav"
    )

    shutil.copy2(
        candidate,
        dest,
    )

    library[
        "voices"
    ][vid] = {
        "name": name,
        "file": (
            f"voices/{vid}.wav"
        ),
        "transcript": (
            STATE.get(
                "audition_text"
            )
            or ""
        ),
        "direction": (
            STATE.get(
                "audition_direction"
            )
            or ""
        ),
        "profile_id": (
            STATE.get(
                "audition_profile_id"
            )
        ),
        "source_type": (
            "voice_design"
        ),
        "created": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
    }

    library[
        "active_voice_id"
    ] = vid

    save_library()

    refresh_saved_voices(
        select_id=vid
    )

    save_design_btn.disabled = True
    save_design_btn.description = (
        "✓ Saved"
    )

    design_status.value = (
        f"<b>🔒 Saved:</b> "
        f"{name}"
    )


design_btn.on_click(
    on_design
)

save_design_btn.on_click(
    on_save_design
)

design_tab = widgets.VBox(
    [
        widgets.HTML(
            '<div class="vox-section">'
            "Choose a test voice profile"
            "</div>"
        ),
        profile_select,
        profile_info,
        widgets.HTML(
            '<div class="vox-section">'
            "Short audition text"
            "</div>"
        ),
        design_audition_text,
        design_name,
        design_btn,
        design_status,
        design_player_box,
    ]
)

# ------------------------------------------------------------
# VOICE CLONE TAB
# ------------------------------------------------------------

# Voice Clone accepts remote URLs only in v15.
# Supported by the existing downloader: YouTube, TikTok, direct media URLs,
# Google Drive and MEGA (subject to site access/cookies).
clone_source_mode = widgets.ToggleButtons(
    options=[
        (
            "Upload file",
            "upload",
        ),
        (
            "Remote upload",
            "url",
        ),
    ],
    value="upload",
    description="Source",
)

clone_upload = widgets.FileUpload(
    accept=(
        ".wav,.mp3,.m4a,.aac,.flac,"
        ".ogg,.opus,.wma,"
        ".mp4,.mkv,.webm,.mov,.avi,"
        ".mpeg,.mpg,.m4v"
    ),
    multiple=False,
    description="Choose audio/video",
)

clone_url = widgets.Text(
    value="",
    description="Remote URL",
    placeholder=(
        "YouTube / TikTok / Google Drive / MEGA / direct media URL"
    ),
    layout=widgets.Layout(
        width="100%"
    ),
)

cookie_upload = widgets.FileUpload(
    accept=".txt",
    multiple=False,
    description="cookies.txt",
)

save_cookie_btn = widgets.Button(
    description="Use cookie file",
    layout=widgets.Layout(
        width="150px"
    ),
)

clear_cookie_btn = widgets.Button(
    description="Clear cookies",
    layout=widgets.Layout(
        width="125px"
    ),
)

cookie_status = widgets.HTML()

remote_user_agent = widgets.Text(
    value="",
    description="User-Agent",
    placeholder="optional",
    layout=widgets.Layout(
        width="100%"
    ),
)

remote_referer = widgets.Text(
    value="",
    description="Referer",
    placeholder="optional",
    layout=widgets.Layout(
        width="100%"
    ),
)

extract_vocals = widgets.Checkbox(
    value=False,
    description=(
        "Remove music / SFX "
        "with Demucs vocal isolation"
    ),
    indent=False,
)

reduce_noise = widgets.Checkbox(
    value=True,
    description=(
        "Reduce residual "
        "background noise"
    ),
    indent=False,
)

noise_amount = widgets.FloatSlider(
    value=0.70,
    min=0.20,
    max=1.00,
    step=0.05,
    description="Noise remove",
    continuous_update=False,
    layout=widgets.Layout(
        width="560px"
    ),
)

clone_start = widgets.FloatText(
    value=0.0,
    description="Start sec",
    layout=widgets.Layout(
        width="280px"
    ),
)

clone_max = widgets.FloatSlider(
    value=20.0,
    min=5.0,
    max=28.0,
    step=1.0,
    description="Max seconds",
    continuous_update=False,
    layout=widgets.Layout(
        width="560px"
    ),
)

whisper_model_info = widgets.HTML(
    "<div class='vox-card'>"
    "<b>Burmese GPU transcription</b>"
    "<br>"
    "<span class='vox-note'>"
    "<b>Auto</b> uses large-v3 only when there is comfortable disk headroom; "
    "otherwise it switches to a ~244M Whisper-Small Burmese fine-tune. "
    "The primary compact model is <b>thantzinphyo/Whisper-Small-ASR</b>, "
    "whose model card reports 5.74% CER and 28.22% WER on its unseen-speaker "
    "test. If that repo cannot be reached, the notebook automatically tries "
    "other public Burmese Whisper-Small fallbacks. "
    "Built-in public models explicitly ignore stale HF login tokens."
    "</span>"
    "</div>"
)

burmese_asr_mode = widgets.Dropdown(
    options=[
        (
            "Auto — best model that fits current disk",
            "auto",
        ),
        (
            "Compact — Burmese Whisper Small (~0.5–1 GB)",
            "compact",
        ),
        (
            "Large — Whisper large-v3 Myanmar (~6.2 GB)",
            "large",
        ),
        (
            "Whisper v3 PEFT/LoRA adapter",
            "lora",
        ),
    ],
    value="auto",
    description="Burmese ASR",
    layout=widgets.Layout(
        width="760px"
    ),
)

burmese_whisper_model_id = widgets.Text(
    value="",
    description="LoRA repo",
    placeholder=(
        "your-user/whisper-large-v3-burmese-lora"
    ),
    layout=widgets.Layout(
        width="100%",
        display="none",
    ),
)

burmese_model_choice_note = widgets.HTML()

burmese_accuracy_note = widgets.HTML(
    "<div class='vox-note'>"
    "<b>For best Burmese transcript:</b> use a clean 10–20 second segment "
    "with one speaker, enable vocal isolation/noise reduction when music is "
    "present, and correct the transcript box before saving the clone. "
    "Real TikTok/YouTube speech can differ from the fine-tuning data, so "
    "proper nouns, English brand names, slang, music and compression can "
    "still produce ASR mistakes."
    "</div>"
)


def _on_burmese_asr_mode_change(
    change=None,
):
    mode = (
        burmese_asr_mode.value
    )

    burmese_whisper_model_id.layout.display = (
        "block"
        if mode == "lora"
        else "none"
    )

    free_gb = working_disk_free_gb()

    if mode == "auto":
        burmese_model_choice_note.value = (
            "<div class='vox-note'>"
            f"Current free disk: <b>{free_gb:.1f} GB</b>. "
            "Auto uses large-v3 at ≥12 GB free; otherwise Compact."
            "</div>"
        )

    elif mode == "compact":
        burmese_model_choice_note.value = (
            "<div class='vox-note'>"
            f"Compact model: <b>{DEFAULT_BURMESE_COMPACT_MODEL}</b>"
            "</div>"
        )

    elif mode == "large":
        burmese_model_choice_note.value = (
            "<div class='vox-note'>"
            f"Large model: <b>{DEFAULT_BURMESE_LARGE_MODEL}</b>. "
            "Requires substantial disk space."
            "</div>"
        )

    else:
        burmese_model_choice_note.value = (
            "<div class='vox-note'>"
            "Paste a real Hugging Face PEFT/LoRA adapter repository."
            "</div>"
        )


burmese_asr_mode.observe(
    _on_burmese_asr_mode_change,
    names="value",
)

_on_burmese_asr_mode_change()


whisper_language = widgets.Dropdown(
    options=WHISPER_LANGUAGE_OPTIONS,
    value="my",
    description="Ref language",
    layout=widgets.Layout(
        width="620px"
    ),
)

whisper_custom_language = widgets.Text(
    value="",
    description="Custom code",
    placeholder=(
        "Whisper language code, e.g. tl, sw, ca"
    ),
    disabled=True,
    layout=widgets.Layout(
        width="620px"
    ),
)


def _on_whisper_language_change(
    change=None,
):
    whisper_custom_language.disabled = (
        whisper_language.value
        != "custom"
    )


whisper_language.observe(
    _on_whisper_language_change,
    names="value",
)

_on_whisper_language_change()


def selected_whisper_language():
    value = whisper_language.value

    if value == "custom":
        custom = (
            whisper_custom_language.value
            .strip()
            .lower()
        )

        if not custom:
            raise ValueError(
                "Enter a Whisper language code "
                "in Custom code."
            )

        return custom

    return value


clone_name = widgets.Text(
    value="My Cloned Voice",
    description="Voice name",
    layout=widgets.Layout(
        width="640px"
    ),
)

process_clone_btn = widgets.Button(
    description=(
        "✨ Prepare & transcribe clone"
    ),
    button_style="success",
    layout=widgets.Layout(
        width="290px",
        height="48px",
    ),
)

save_clone_btn = widgets.Button(
    description=(
        "🔒 Save clone to library"
    ),
    button_style="warning",
    disabled=True,
    layout=widgets.Layout(
        width="250px",
        height="48px",
    ),
)

clone_status = widgets.HTML()

clone_transcript = widgets.Textarea(
    value="",
    placeholder=(
        "Whisper transcript appears here. "
        "Correct it before saving if necessary."
    ),
    layout=widgets.Layout(
        width="100%",
        height="150px",
    ),
)

clone_output = widgets.Output()

clone_test_text = widgets.Textarea(
    value=(
        "မင်္ဂလာပါ။ ဒီအသံကို clone လုပ်ထားပြီး "
        "သဘာဝကျကျ ပြောနိုင်မနိုင် စမ်းသပ်နေပါတယ်။"
    ),
    placeholder=(
        "Enter text to test the prepared cloned voice…"
    ),
    layout=widgets.Layout(
        width="100%",
        height="100px",
    ),
)

test_clone_btn = widgets.Button(
    description="▶ Test cloned voice",
    button_style="info",
    disabled=True,
    layout=widgets.Layout(
        width="220px",
        height="44px",
    ),
)

clone_test_status = widgets.HTML()
clone_test_output = widgets.Output()


def _test_clone_job():
    try:
        ref = STATE.get(
            "clone_path"
        )

        if (
            not ref
            or not Path(ref).exists()
        ):
            raise ValueError(
                "Prepare the clone reference first."
            )

        transcript = (
            clone_transcript.value
            .strip()
        )

        if not transcript:
            raise ValueError(
                "Reference transcript is empty. "
                "Correct/save the transcript first."
            )

        test_text = sanitize_tts_text(
            clone_test_text.value
        )

        if not has_speakable_content(
            test_text
        ):
            raise ValueError(
                "Enter test text for the cloned voice."
            )

        if STATE.get(
            "generation_busy"
        ):
            raise RuntimeError(
                "Long-form generation is currently using the GPUs."
            )

        if STATE.get(
            "clone_busy"
        ):
            raise RuntimeError(
                "Clone preparation/transcription is still running."
            )

        clone_test_status.value = (
            "<b>Generating cloned-voice test…</b>"
        )

        stamp = time.strftime(
            "%Y%m%d_%H%M%S"
        )

        out = (
            GENERATED_DIR
            / (
                f"clone_test_"
                f"{stamp}_"
                f"{uuid.uuid4().hex[:5]}"
                f".wav"
            )
        )

        results, _ = _VOX_SERVICE.run(
            [
                {
                    "chunk_index": 0,
                    "output_path": str(
                        out
                    ),
                    "kwargs": {
                        "text": test_text,
                        "reference_wav_path": str(
                            ref
                        ),
                        "prompt_wav_path": str(
                            ref
                        ),
                        "prompt_text": transcript,
                        "cfg_value": float(
                            cfg_slider.value
                        ),
                        "inference_timesteps": int(
                            steps_slider.value
                        ),
                        "retry_badcase": True,
                        "normalize": False,
                    },
                }
            ]
        )

        result = results[0]

        clone_test_output.clear_output(
            wait=True
        )

        clone_test_output.append_display_data(
            make_static_audio_card(
                path=out,
                title="🎧 Cloned voice test",
                mime="audio/wav",
            )
        )

        clone_test_status.value = (
            f"<b>✓ Clone test ready</b> • "
            f"GPU {result.get('gpu')}"
        )

    except Exception as e:
        clone_test_status.value = (
            "<span style='color:#ff5f56'>"
            "<b>Test failed:</b> "
            f"{str(e)}"
            "</span>"
        )

    finally:
        test_clone_btn.disabled = False


def on_test_clone(_):
    if STATE.get(
        "clone_busy"
    ):
        clone_test_status.value = (
            "<span style='color:#b7791f'>"
            "Wait for clone preparation/transcription to finish."
            "</span>"
        )
        return

    test_clone_btn.disabled = True

    threading.Thread(
        target=_test_clone_job,
        daemon=True,
    ).start()


test_clone_btn.on_click(
    on_test_clone
)


def on_save_cookie(_):
    try:
        path = save_cookie_upload(
            cookie_upload
        )

        if not path:
            raise ValueError(
                "Choose a Netscape "
                "cookies.txt file first."
            )

        STATE[
            "cookie_file"
        ] = path

        cookie_status.value = (
            "<span style='color:#2ea043'>"
            "✓ Cookie file loaded "
            "for this session."
            "</span>"
        )

    except Exception as e:
        cookie_status.value = (
            "<span style='color:#ff5f56'>"
            f"{str(e)}"
            "</span>"
        )


def on_clear_cookie(_):
    STATE[
        "cookie_file"
    ] = None

    path = (
        COOKIE_DIR
        / "cookies.txt"
    )

    if path.exists():
        try:
            path.unlink()
        except Exception:
            pass

    cookie_status.value = (
        "Cookies cleared."
    )


save_cookie_btn.on_click(
    on_save_cookie
)

clear_cookie_btn.on_click(
    on_clear_cookie
)


def set_clone_status(
    text,
    color="#8c8c8c",
):
    clone_status.value = (
        "<div class='vox-card' "
        f"style='color:{color}'>"
        f"{text}"
        "</div>"
    )


def _clone_pipeline():
    job_dir = (
        IMPORT_DIR
        / (
            f"clone_"
            f"{int(time.time())}_"
            f"{uuid.uuid4().hex[:6]}"
        )
    )

    job_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    try:
        cleanup_clone_temp_storage(
            preserve_dir=job_dir,
            status_callback=(
                lambda message: set_clone_status(
                    "0/5 • " + str(message)
                )
            ),
        )

        if clone_source_mode.value == "upload":
            set_clone_status(
                "1/5 • Reading uploaded audio/video…"
            )

            source = save_upload_to_dir(
                clone_upload,
                job_dir,
                fallback_name=(
                    "source_media.bin"
                ),
            )

        else:
            remote_url = (
                clone_url.value
                .strip()
            )

            if not remote_url:
                raise ValueError(
                    "Paste a remote media URL first."
                )

            set_clone_status(
                "1/5 • Downloading remote media…"
            )

            source = download_remote_media(
                remote_url,
                job_dir,
                cookie_file=(
                    STATE.get(
                        "cookie_file"
                    )
                ),
                user_agent=(
                    remote_user_agent.value
                ),
                referer=(
                    remote_referer.value
                ),
            )

        STATE[
            "clone_source"
        ] = str(source)

        set_clone_status(
            "2/5 • Converting to mono WAV "
            "and selecting reference segment…"
        )

        normalized = (
            job_dir
            / "reference_normalized.wav"
        )

        normalize_media_to_wav(
            source_path=source,
            output_path=normalized,
            start_seconds=max(
                0.0,
                float(
                    clone_start.value
                ),
            ),
            max_seconds=max(
                1.0,
                float(
                    clone_max.value
                ),
            ),
        )

        working = str(
            normalized
        )

        if extract_vocals.value:
            set_clone_status(
                "3/5 • Removing music / SFX "
                "with Demucs on CPU…"
            )

            try:
                working = extract_vocals_demucs(
                    working,
                    job_dir,
                )

            except Exception as e:
                set_clone_status(
                    "Demucs failed; continuing "
                    "with normalized source. "
                    f"{str(e)}",
                    color="#b7791f",
                )

                working = str(
                    normalized
                )

        else:
            set_clone_status(
                "3/5 • Music/SFX "
                "isolation skipped."
            )

        if reduce_noise.value:
            set_clone_status(
                "4/5 • Reducing residual "
                "background noise…"
            )

            denoised = (
                job_dir
                / "denoised.wav"
            )

            reduce_residual_noise(
                working,
                denoised,
                amount=float(
                    noise_amount.value
                ),
            )

            working = str(
                denoised
            )

        final_ref = (
            job_dir
            / "clone_reference.wav"
        )

        normalize_media_to_wav(
            source_path=working,
            output_path=final_ref,
            start_seconds=0,
            max_seconds=max(
                1.0,
                float(
                    clone_max.value
                ),
            ),
        )

        wav, sr = sf.read(
            str(final_ref),
            dtype="float32",
            always_2d=False,
        )

        wav = trim_edge_silence(
            wav,
            sr,
            threshold_db=-45.0,
            keep_ms=40,
        )

        sf.write(
            str(final_ref),
            wav,
            int(sr),
            subtype="PCM_16",
        )

        selected_language = (
            selected_whisper_language()
        )

        def whisper_status(
            message,
        ):
            set_clone_status(
                "5/5 • "
                + str(message)
            )

        if selected_language == "my":
            if burmese_asr_mode.value == "lora":
                set_clone_status(
                    "5/5 • Validating/downloading Burmese Whisper "
                    "PEFT/LoRA…"
                )
            elif burmese_asr_mode.value == "large":
                set_clone_status(
                    "5/5 • Preparing Burmese Whisper large-v3 ASR…"
                )
            elif burmese_asr_mode.value == "compact":
                set_clone_status(
                    "5/5 • Preparing compact Burmese Whisper-Small ASR…"
                )
            else:
                set_clone_status(
                    "5/5 • Auto-selecting Burmese ASR for available disk…"
                )
        else:
            set_clone_status(
                "5/5 • Preparing GPU transcription with "
                "Whisper large-v3…"
            )

        selected_burmese_mode = (
            burmese_asr_mode.value
        )

        selected_burmese_repo = (
            burmese_whisper_model_id.value
            .strip()
        )

        if (
            selected_language == "my"
            and selected_burmese_mode == "lora"
            and not selected_burmese_repo
        ):
            raise ValueError(
                "LoRA mode is selected, but no Burmese LoRA repo was entered."
            )

        (
            transcript,
            detected,
            language_probability,
            asr_backend,
            asr_model_name,
        ) = transcribe_reference(
            final_ref,
            language=(
                selected_language
            ),
            burmese_mode=(
                selected_burmese_mode
            ),
            burmese_model_repo=(
                selected_burmese_repo
            ),
            status_callback=(
                whisper_status
            ),
        )

        STATE[
            "clone_path"
        ] = str(
            final_ref
        )

        STATE[
            "clone_transcript"
        ] = transcript

        clone_transcript.value = (
            transcript
        )

        clone_output.clear_output(
            wait=True
        )

        probability_text = ""

        if (
            language_probability
            is not None
        ):
            probability_text = (
                f" "
                f"({float(language_probability) * 100:.1f}%)"
            )

        clone_output.append_stdout(
            "✓ Prepared clone reference\n"
            f"Selected ref language: "
            f"{selected_language}\n"
            f"Detected language: "
            f"{detected or 'unknown'}"
            f"{probability_text}\n"
            f"ASR backend: {asr_backend}\n"
            f"ASR model: {asr_model_name} • GPU 1\n\n"
        )

        clone_output.append_display_data(
            make_static_audio_card(
                path=final_ref,
                title="🧬 Prepared clone reference",
                mime="audio/wav",
            )
        )

        save_clone_btn.disabled = False
        test_clone_btn.disabled = False

        set_clone_status(
            "✓ Clone reference is ready. "
            "Listen, correct the transcript "
            "if needed, then save it.",
            color="#2ea043",
        )

    except Exception as e:
        set_clone_status(
            "<b>Clone preparation failed:</b> "
            f"{str(e)}",
            color="#ff5f56",
        )

    finally:
        STATE[
            "clone_busy"
        ] = False

        process_clone_btn.disabled = False


def on_process_clone(_):
    if STATE[
        "clone_busy"
    ]:
        return

    if STATE.get(
        "generation_busy"
    ):
        set_clone_status(
            "Wait for generation to finish or press Stop first. "
            "GPU 1 must be idle before Whisper can borrow it.",
            color="#b7791f",
        )
        return

    if STATE.get(
        "audition_busy"
    ):
        set_clone_status(
            "Wait for the current Voice Design test to finish first.",
            color="#b7791f",
        )
        return

    STATE[
        "clone_busy"
    ] = True

    clone_disk_info.value = (
        "<div class='vox-note'>"
        f"Kaggle working disk free: "
        f"<b>{working_disk_free_gb():.1f} GB</b>"
        "</div>"
    )

    process_clone_btn.disabled = True
    save_clone_btn.disabled = True
    test_clone_btn.disabled = True

    clone_test_output.clear_output(
        wait=True
    )
    clone_test_status.value = ""

    threading.Thread(
        target=_clone_pipeline,
        daemon=True,
    ).start()


def on_save_clone(_):
    ref = STATE.get(
        "clone_path"
    )

    if (
        not ref
        or not Path(
            ref
        ).exists()
    ):
        set_clone_status(
            "Prepare a clone "
            "reference first.",
            color="#ff5f56",
        )
        return

    transcript = (
        clone_transcript.value
        .strip()
    )

    if not transcript:
        set_clone_status(
            "Transcript is empty. "
            "Enter the exact spoken words "
            "before saving.",
            color="#b7791f",
        )
        return

    name = (
        clone_name.value.strip()
        or "Cloned Voice"
    )

    vid = (
        uuid.uuid4().hex[:12]
    )

    dest = (
        VOICE_DIR
        / f"{vid}.wav"
    )

    shutil.copy2(
        ref,
        dest,
    )

    library[
        "voices"
    ][vid] = {
        "name": name,
        "file": (
            f"voices/{vid}.wav"
        ),
        "transcript": transcript,
        "direction": "",
        "profile_id": None,
        "source_type": (
            "voice_clone"
        ),
        "source": (
            STATE.get(
                "clone_source",
                "",
            )
        ),
        "created": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
    }

    library[
        "active_voice_id"
    ] = vid

    save_library()
    refresh_saved_voices(
        select_id=vid
    )

    set_clone_status(
        f"🔒 Saved cloned voice: "
        f"{name}",
        color="#2ea043",
    )


process_clone_btn.on_click(
    on_process_clone
)

save_clone_btn.on_click(
    on_save_clone
)

cookie_advanced = widgets.Accordion(
    children=[
        widgets.VBox(
            [
                widgets.HTML(
                    "<div class='vox-note'>"
                    "Optional for sites requiring "
                    "your own authenticated session. "
                    "Upload a Netscape-format cookies.txt. "
                    "Do not share the cookie file."
                    "</div>"
                ),
                widgets.HBox(
                    [
                        cookie_upload,
                        save_cookie_btn,
                        clear_cookie_btn,
                    ]
                ),
                cookie_status,
                remote_user_agent,
                remote_referer,
            ]
        )
    ],
    selected_index=None,
)

cookie_advanced.set_title(
    0,
    "Advanced remote-download cookies / headers",
)


clone_upload_box = widgets.VBox(
    [
        widgets.HTML(
            "<b>Upload file</b>"
        ),
        clone_upload,
    ]
)

clone_remote_box = widgets.VBox(
    [
        widgets.HTML(
            "<b>Remote URL</b>"
        ),
        clone_url,
        cookie_advanced,
    ]
)


def _refresh_clone_source_ui(
    change=None,
):
    is_remote = (
        clone_source_mode.value
        == "url"
    )

    clone_upload_box.layout.display = (
        "none"
        if is_remote
        else "block"
    )

    clone_remote_box.layout.display = (
        "block"
        if is_remote
        else "none"
    )


clone_source_mode.observe(
    _refresh_clone_source_ui,
    names="value",
)

_refresh_clone_source_ui()


clone_disk_info = widgets.HTML(
    value=(
        "<div class='vox-note'>"
        f"Kaggle working disk free: "
        f"<b>{working_disk_free_gb():.1f} GB</b>. "
        "v16 automatically removes incomplete downloads and temporary "
        "chunk folders. Auto Burmese ASR falls back to a compact ~1 GB model "
        "when the large model does not safely fit."
        "</div>"
    )
)

cleanup_advanced = widgets.Accordion(
    children=[
        widgets.VBox(
            [
                clone_disk_info,
                extract_vocals,
                reduce_noise,
                noise_amount,
                clone_start,
                clone_max,
                whisper_model_info,
                burmese_asr_mode,
                burmese_whisper_model_id,
                burmese_model_choice_note,
                burmese_accuracy_note,
                whisper_language,
                whisper_custom_language,
            ]
        )
    ],
    selected_index=None,
)

cleanup_advanced.set_title(
    0,
    "Cleanup / transcription settings",
)

clone_tab = widgets.VBox(
    [
        widgets.HTML(
            '<div class="vox-section">'
            "Clone a voice from audio/video"
            "</div>"
        ),
        widgets.HTML(
            '<div class="vox-note">'
            "Choose a local audio/video file or Remote upload. "
            "The URL field appears only when Remote upload is selected. "
            "A clean single-speaker sample works best."
            "</div>"
        ),
        clone_source_mode,
        clone_upload_box,
        clone_remote_box,
        cleanup_advanced,
        clone_name,
        widgets.HBox(
            [
                process_clone_btn,
                save_clone_btn,
            ]
        ),
        clone_status,
        widgets.HTML(
            '<div class="vox-section">'
            "Reference transcript"
            "</div>"
        ),
        clone_transcript,
        clone_output,

        widgets.HTML(
            '<div class="vox-section">'
            "Test cloned voice"
            "</div>"
        ),
        widgets.HTML(
            '<div class="vox-note">'
            "After the reference and transcript are ready, generate a short "
            "test before saving the voice permanently."
            "</div>"
        ),
        clone_test_text,
        test_clone_btn,
        clone_test_status,
        clone_test_output,
    ]
)

# ------------------------------------------------------------
# LIBRARY / PERSISTENCE TAB
# ------------------------------------------------------------

dataset_handle = widgets.Text(
    value=(
        VOICE_DATASET_HANDLE_DEFAULT
    ),
    description="Dataset",
    placeholder=(
        "username/dataset-slug"
    ),
    layout=widgets.Layout(
        width="640px"
    ),
)

sync_btn = widgets.Button(
    description=(
        "☁ Sync voice library"
    ),
    button_style="info",
    layout=widgets.Layout(
        width="220px",
        height="44px",
    ),
)

sync_status = widgets.HTML()


def on_sync(_):
    sync_btn.disabled = True

    try:
        sync_status.value = (
            "Uploading current "
            "library…"
        )

        sync_library_to_dataset(
            dataset_handle.value
        )

        sync_status.value = (
            "<span style='color:#2ea043'>"
            "✓ Kaggle Dataset synced. "
            "Attach its latest version "
            "as notebook Input next time."
            "</span>"
        )

    except Exception as e:
        sync_status.value = (
            "<span style='color:#ff5f56'>"
            "Sync failed: "
            f"{str(e)}"
            "</span>"
        )

    finally:
        sync_btn.disabled = False


sync_btn.on_click(
    on_sync
)

library_tab = widgets.VBox(
    [
        widgets.HTML(
            '<div class="vox-section">'
            "Permanent voice library"
            "</div>"
        ),
        widgets.HBox(
            [
                saved_voice,
                play_selected_btn,
                delete_selected_btn,
            ]
        ),
        library_output,
        widgets.HTML(
            '<div class="vox-section">'
            "Persistent Kaggle Dataset"
            "</div>"
        ),
        widgets.HTML(
            '<div class="vox-note">'
            "Sync library.json and permanent voice WAVs "
            "to a Kaggle Dataset. Generated files, imported "
            "media and cookies are not uploaded."
            "</div>"
        ),
        dataset_handle,
        sync_btn,
        sync_status,
    ]
)

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

refresh_saved_voices()

if restored_from:
    sync_status.value = (
        "<span style='color:#2ea043'>"
        "✓ Restored voice library from: "
        f"{restored_from}"
        "</span>"
    )

tabs = widgets.Tab(
    children=[
        generate_tab,
        design_tab,
        clone_tab,
        library_tab,
    ],
    layout=widgets.Layout(
        width="100%"
    ),
)

tabs.set_title(
    0,
    "⚡ Generate",
)

tabs.set_title(
    1,
    "🎭 Voice Design",
)

tabs.set_title(
    2,
    "🧬 Voice Clone",
)

tabs.set_title(
    3,
    "💾 Library",
)

display(
    studio_css
)

display(
    header
)

display(
    tabs
)


Loading VoxCPM2 on T4 GPUs sequentially...
Starting VoxCPM2 workers sequentially to avoid model-load/cache contention...
\n→ Starting physical GPU 0...
  worker PID 157
  • GPU 0: starting child process
  … GPU 0: starting child process • 0s elapsed
  • GPU 0: torch/soundfile imported
  … GPU 0: torch/soundfile imported • 10s elapsed


voxcpm_model_path: /kaggle/working/VoxCPM2, zipenhancer_model_path: None, enable_denoiser: False


  • GPU 0: VoxCPM.from_pretrained


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Loading AudioVAE from pytorch: /kaggle/working/VoxCPM2/audiovae.pth
Running on device: cuda, dtype: bfloat16


  … GPU 0: VoxCPM.from_pretrained • 21s elapsed
  … GPU 0: VoxCPM.from_pretrained • 31s elapsed


...

Loading model from safetensors: /kaggle/working/VoxCPM2/model.safetensors


  • GPU 1: model object loaded on CUDA
✓ GPU 1 ready (48000 Hz) after 28s
✓ Both VoxCPM2 workers are ready.
✓ Dual-T4 service ready


Loaded VoxCPM2Model


HTML(value='\n<style>\n.vox-card {\n    background: #181818;\n    color: #ffffff;\n    border: 1px solid #3434…

HTML(value='\n<div class="vox-card">\n  <div class="vox-title">\n    🎙️ VoxCPM2 Voice Studio\n  </div>\n  <div…

## Notes

- **Style fix:** when a delivery instruction is present, the studio uses VoxCPM2 reference-audio-only controllable cloning instead of mixing the style instruction with prompt-text continuation. This prevents text such as “slightly faster” from being treated like narration.
- **Speed:** the Speed control uses FFmpeg `atempo`, so it is deterministic and preserves pitch.
- **Join pause:** generated edge silence is trimmed first; then the requested pause is inserted, so 100 ms/300 ms now has a predictable effect.
- **Clone preprocessing:** Demucs and Whisper run on CPU in a background thread because both T4s are reserved for VoxCPM2.
- Use only media you have permission to download and clone. Cookies stay in a separate private working folder and are never included by the voice-library sync. Imported source media and generated outputs are also excluded from the persistent Dataset.


### VoxCPM package compatibility
This revision inspects `model._generate` inside each GPU worker. Unsupported keyword arguments are removed automatically. On older builds without native `seed=`, a requested deterministic seed is applied directly to Python, NumPy and PyTorch RNGs instead.


### Progressive / live listening
Long generation now exposes completed chunks immediately. Because a normal WAV file is finalized with its length in the header and Kaggle/Jupyter rich output is served as a static object, the studio uses **ordered segmented streaming**: Part 1 appears as soon as it finishes, then Part 2, Part 3, and so on while both GPUs keep synthesizing later chunks. The final combined WAV is still created at the end.


### Streaming player v4
The live player now keeps one cumulative preview. After each **contiguous** chunk is ready, the notebook rebuilds a small 64-kbps MP3 preview and updates the same browser audio player while preserving its playback position. If playback reaches the currently generated edge first, the UI shows **“Buffering — still generating…”**; when another chunk arrives, the player attempts to resume automatically. Individual completed chunks are also shown as fallback audio players. The final downloadable result remains a full-quality WAV.
